# 12-03 TabPFN Final Test mit drei Targets, validiertem Threshold und optionalem Thinking-Vergleich

Dieses Notebook ist der echte Zukunftstest für TabPFN. Es lädt die in `12-02` gewählte normale TabPFN-Konfiguration und den dort validierten Frank-&-Hall-Threshold. Auf 2024/2025 wird nichts mehr optimiert.

Die finale sportliche Rankingbewertung bleibt `score_topk_raw_sum` innerhalb jeder Etappe. Die binären Top-K-Wahrscheinlichkeiten werden zusätzlich wie im EBM-Pfad für Klassifikationsmetriken und Konfusionsmatrizen ausgewertet.


## Methodische Langfassung: Vom binären Classifier zur finalen Testauswertung

TabPFN wird in diesem Projekt nicht als direktes Rankingmodell trainiert. Stattdessen werden drei binäre, kumulative Fragen gestellt: `Top5`, `Top10` und `Top20`. Diese Struktur entspricht der Frank-&-Hall-Idee: Ordinale Information wird über mehrere binäre Schwellenwertmodelle abgebildet.

**Warum `12-03` nicht optimiert:** Der Testzeitraum 2024/2025 ist die spätere Realität. Jede Threshold-Suche auf diesen Daten wäre Leakage. Deshalb lädt `12-03` den in `12-02` auf 2023 validierten Threshold und wendet ihn unverändert an. Falls der Threshold fehlt, warnt das Notebook und kann native Metriken sowie Rankings berechnen, aber keine validierte Ensemble-Klassifikation berichten.

**Zwei Auswertungsarten:** Die Klassifikationsauswertung beantwortet Fragen wie: Wie gut erkennt der Score echte Top10-Fahrer bei einem festen Threshold? Dafür sind Konfusionsmatrizen, Precision, Recall, F1, ROC-AUC und Average Precision relevant. Die Rankingauswertung beantwortet dagegen: Wie gut sortiert der Score alle Fahrer innerhalb einer Etappe? Dafür werden NDCG@5/10/20, Winner-Hit-Rates und MAP berechnet.

**Keine Differenzwahrscheinlichkeiten als Hauptaussage:** Die drei TabPFN-Modelle werden separat trainiert. Deshalb ist `p_top10_raw - p_top5_raw` keine garantiert gültige Wahrscheinlichkeit für Rang 6-10. Solche Differenzen können sogar negativ werden, wenn die kumulativen Wahrscheinlichkeiten nicht monoton sind. Dieses Notebook dokumentiert Monotonieverletzungen und nutzt für interpretierbare Klassen nur zwei robuste Sichten: den erzwungenen Ranking-Bucket aus dem stage-weiten Ranking und eine sequenzielle Frank-&-Hall-Klasse mit Threshold `0.5`.

**Thinking-Vergleich:** Thinking bleibt ein optionaler finaler Vergleich. Er wird nicht für Validierung oder Threshold-Wahl verwendet. Ein Thinking-Cache gilt nur dann als echter Thinking-Lauf, wenn Fingerprint, Parameter, Timeout und Runtime-Sidecar exakt passen.


## Metrik- und Output-Legende für den finalen Test

Klassifikation:

- `top5_native_0_5`, `top10_native_0_5`, `top20_native_0_5`: Einzelkanäle mit festem Threshold `0.5`.
- `frank_hall_top10_native_0_5`: naive Ensemble-Baseline mit `score_topk_raw_sum >= 0.5`.
- `frank_hall_top10_validated_threshold`: Ensemble-Klassifikation mit dem in `12-02` validierten Threshold.
- `tabpfn_final_confusion_matrices.csv`: 2x2-Konfusionsmatrizen für binäre Testentscheidungen.
- `tabpfn_final_classification_metrics.csv`: Precision, Recall, F1, ROC-AUC und Average Precision.

Ranking:

- `score_topk_raw_sum`: Summe aus `p_top5_raw`, `p_top10_raw` und `p_top20_raw`.
- `rank_topk_raw_sum`: stage-weites Ranking nach diesem Score.
- `predicted_rank_bucket`: Top5, Rang 6-10, Rang 11-20 oder außerhalb Top20 aus dem Ranking, nicht aus Differenzwahrscheinlichkeiten.
- `NDCG@5/10/20`, Winner-Hit-Rates und `MAP`: finale sportliche Rankingmetriken.

Diagnostics:

- `is_cumulative_probability_monotonic`: prüft, ob `p_top5_raw <= p_top10_raw <= p_top20_raw` gilt.
- `fh_seq_class_label_0_5`: sequenzielle Frank-&-Hall-Klasse mit Threshold `0.5`.
- Thinking-Vergleich: nur vollständig und fingerprint-validierte Thinking-Läufe werden als Vergleich akzeptiert.


## Imports und Projektpfade

Der erste Schritt richtet die Arbeitsumgebung ein und sucht die Projektwurzel direkt im Notebook. Dadurch sind alle folgenden Pfade explizit und nachvollziehbar.

### Erwarteter Output

Die Pfadprüfung soll zeigen, wohin finale Predictions, Case-Study-Tabellen und kompakte Modellvergleichsdateien geschrieben werden. Damit wird der spätere Vergleich mit EBM und XGBoost nachvollziehbar.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
import hashlib
import json
import os
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Die Projektwurzel wird sichtbar gesucht, ohne eine Hilfsfunktion auszulagern.
start_path = Path.cwd()
PROJECT_ROOT = None
for candidate in [start_path, *start_path.parents]:
    has_model_data = (candidate / "data" / "model_data").exists()
    has_notebooks = (candidate / "src" / "Notebooks").exists()
    if has_model_data and has_notebooks:
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Projektwurzel mit data/model_data und src/Notebooks nicht gefunden.")

MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "data" / "models"
TABPFN_DIR = PROCESSED_DIR / "tabpfn"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
STANDARD_MODEL_DIR = MODEL_DIR / "tabpfn_final_standard_models"
STANDARD_MODEL_MANIFEST_PATH = STANDARD_MODEL_DIR / "tabpfn_standard_model_manifest.json"
STANDARD_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MODEL_DATA_DIR: {MODEL_DATA_DIR}")

path_overview = pd.DataFrame([
    {"name": "PROJECT_ROOT", "path": str(PROJECT_ROOT), "exists": PROJECT_ROOT.exists()},
    {"name": "MODEL_DATA_DIR", "path": str(MODEL_DATA_DIR), "exists": MODEL_DATA_DIR.exists()},
    {"name": "TABPFN_DIR", "path": str(TABPFN_DIR), "exists": TABPFN_DIR.exists()},
    {"name": "MODEL_DIR", "path": str(MODEL_DIR), "exists": MODEL_DIR.exists()},
])

print("=" * 88)
print("TABPFN 12-03: Pfadprüfung und finaler Arbeitskontext")
print("=" * 88)
display(path_overview)


PROJECT_ROOT: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction
MODEL_DATA_DIR: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction/data/model_data
TABPFN 12-03: Pfadprüfung und finaler Arbeitskontext


,name,path,exists
0,PROJECT_ROOT,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
1,MODEL_DATA_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
2,TABPFN_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
3,MODEL_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True


## Finale Konfiguration, Features und Output-Ordner

Dieser Abschnitt fixiert die gemeinsame Daten- und Outputstruktur. `RUN_API = False` bleibt als sicherer Default aktiv. Neue API-Läufe entstehen also nur, wenn der Schalter bewusst geändert wird.

**Schutz vor Leakage:** `12-03` lädt später Train+Validation als finalen Trainingskontext und `X_test` nur für die abschließende Bewertung. Es werden keine Hyperparameter mehr anhand des Tests gewählt.

### Warum die finale Konfiguration nochmal dokumentiert wird

Im finalen Test darf keine neue Hyperparameterwahl passieren. Deshalb werden die Settings und Outputpfade vor dem Datenladen sichtbar gemacht. `RUN_API=False` bleibt der Schutz vor unbeabsichtigten API-Läufen; vorhandene Caches können trotzdem gelesen und bewertet werden.

In [ ]:
RUN_API = True  # False: keine API-Läufe; True: API-Läufe werden durchgeführt, wenn sie noch nicht existieren.
FORCE_RERUN = False  # False: bestehende finale Standard-Caches bleiben nutzbar; True: alle finalen Standard-Caches werden neu berechnet.
FORCE_RERUN_VARIANTS = None  # None: keine gezielten Neuberechnungen; Liste von Varianten, die gezielt neu berechnet werden sollen, z.B. ["top5", "top10"].
SEED = 42
TABPFN_CLIENT_N_ESTIMATORS_MAX = 8
THINKING_TIMEOUT_SECONDS = 2400
THINKING_CLIENT_TIMEOUT_SECONDS = 3000
THINKING_MIN_FIT_SECONDS_FOR_WARNING = 30

FEATURE_COLUMNS = [
    "distance",
    "vertical_meters",
    "stage_nr",
    "team_tier",
    "age_at_race",
    "rider_bmi",
    "wind_stability_index",
    "weather_temp_mean",
    "weather_temp_trend",
    "weather_rain_prob_mean",
    "weather_precipitation_mean",
    "weather_humidity_mean",
    "gradient_final_km",
    "lag_rider_points_season",
    "lag_rider_rank_season",
    "lag_race_competitiveness_median",
    "lag_team_power_index",
]

TARGET_CONFIGS = {
    "top5": {
        "label": "Top5",
        "train_file": "y_top5_train.pkl",
        "valid_file": "y_top5_valid.pkl",
        "test_file": "y_top5_test.pkl",
        "score_col": "p_top5_raw",
        "actual_col": "actual_top5",
    },
    "top10": {
        "label": "Top10",
        "train_file": "y_top10_train.pkl",
        "valid_file": "y_top10_valid.pkl",
        "test_file": "y_top10_test.pkl",
        "score_col": "p_top10_raw",
        "actual_col": "actual_top10",
    },
    "top20": {
        "label": "Top20",
        "train_file": "y_top20_train.pkl",
        "valid_file": "y_top20_valid.pkl",
        "test_file": "y_top20_test.pkl",
        "score_col": "p_top20_raw",
        "actual_col": "actual_top20",
    },
}

EXPECTED_SHAPES = {
    "train_final": (178246, 17),
    "test": (17802, 17),
}

TUNING_DIR = TABPFN_DIR / "02_tuning_top10"
RESULT_DIR = TABPFN_DIR / "03_final_three_targets"
PREDICTION_ROOT = RESULT_DIR / "predictions"
CASE_STUDY_DIR = RESULT_DIR / "case_studies"
THINKING_DIR = RESULT_DIR / "thinking_comparison"
for output_path in [RESULT_DIR, PREDICTION_ROOT, CASE_STUDY_DIR, THINKING_DIR]:
    output_path.mkdir(parents=True, exist_ok=True)

print(f"RUN_API: {RUN_API}")
print(f"RESULT_DIR: {RESULT_DIR}")

config_summary = pd.DataFrame([
    {"setting": "RUN_API", "value": RUN_API, "interpretation": "sicherer Default; keine externen API-Läufe beim Öffnen"},
    {"setting": "FORCE_RERUN", "value": FORCE_RERUN, "interpretation": "bestehende finale Standard-Caches bleiben nutzbar"},
    {"setting": "FORCE_RERUN_VARIANTS", "value": ", ".join(FORCE_RERUN_VARIANTS) if FORCE_RERUN_VARIANTS else "", "interpretation": "gezieltes Neurechnen einzelner Varianten"},
    {"setting": "Thinking timeout", "value": THINKING_TIMEOUT_SECONDS, "interpretation": "serverseitiger Thinking-Timeout in Sekunden"},
    {"setting": "Client timeout", "value": THINKING_CLIENT_TIMEOUT_SECONDS, "interpretation": "Client-Timeout muss größer als Thinking-Timeout sein"},
    {"setting": "n_estimators_limit", "value": TABPFN_CLIENT_N_ESTIMATORS_MAX, "interpretation": "tabpfn_client/API-Limit"},
    {"setting": "RESULT_DIR", "value": str(RESULT_DIR), "interpretation": "detaillierte finale TabPFN-Outputs"},
    {"setting": "PREDICTION_ROOT", "value": str(PREDICTION_ROOT), "interpretation": "Target- und Varianten-Predictions"},
    {"setting": "STANDARD_MODEL_DIR", "value": str(STANDARD_MODEL_DIR), "interpretation": "exportierte Standardmodelle fuer 12-04 SHAP"},
    {"setting": "STANDARD_MODEL_MANIFEST_PATH", "value": str(STANDARD_MODEL_MANIFEST_PATH), "interpretation": "Manifest fuer SHAP-Modellartefakte"},
])

target_config_table = pd.DataFrame([
    {"target": target, "label": cfg["label"], "score_col": cfg["score_col"], "actual_col": cfg["actual_col"]}
    for target, cfg in TARGET_CONFIGS.items()
])

print("=" * 88)
print("TABPFN 12-03: Finale Konfiguration und Zielkanäle")
print("=" * 88)
display(config_summary)
display(target_config_table)


RUN_API: True
RESULT_DIR: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction/data/processed/tabpfn/03_final_three_targets
TABPFN 12-03: Finale Konfiguration und Zielkanäle


,setting,value,interpretation
0,RUN_API,True,sicherer Default; keine externen API-Läufe bei...
1,FORCE_RERUN,False,bestehende finale Standard-Caches bleiben nutzbar
2,FORCE_RERUN_VARIANTS,standard,gezieltes Neurechnen einzelner Varianten
3,Thinking timeout,2400,serverseitiger Thinking-Timeout in Sekunden
4,Client timeout,3000,Client-Timeout muss größer als Thinking-Timeou...
5,n_estimators_limit,8,tabpfn_client/API-Limit
6,RESULT_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,detaillierte finale TabPFN-Outputs
7,PREDICTION_ROOT,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,Target- und Varianten-Predictions
8,STANDARD_MODEL_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,exportierte Standardmodelle fuer 12-04 SHAP
9,STANDARD_MODEL_MANIFEST_PATH,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,Manifest fuer SHAP-Modellartefakte


,target,label,score_col,actual_col
0,top5,Top5,p_top5_raw,actual_top5
1,top10,Top10,p_top10_raw,actual_top10
2,top20,Top20,p_top20_raw,actual_top20


## Ausgewählte Top10-Konfiguration laden

Die JSON-Datei aus `12-02` ist die einzige Quelle für die normalen Classifier-Parameter. Thinking-Parameter dürfen hier zusätzlich als finaler Vergleich getestet werden, überschreiben aber die Auswahl nicht.

### Interpretation der geladenen Auswahl

Diese JSON-Datei ist der methodische Vertrag zwischen Validierung und Test. Wenn sie fehlt, nutzt das Notebook nur einen Fallback, damit die Struktur lesbar bleibt. Für eine belastbare Thesis-Auswertung sollte jedoch immer die echte Auswahl aus `12-02` vorhanden sein.

In [3]:
selected_path = TUNING_DIR / "tabpfn_selected_classifier_params.json"
if selected_path.exists():
    with open(selected_path) as f:
        selected = json.load(f)
else:
    selected = {
        "model_family": "TabPFNClassifier",
        "selection_target": "top10",
        "selection_metric": "roc_auc",
        "selection_validation_context": "X_valid, meta_year == 2023",
        "params": {"n_estimators": 4},
        "excluded_from_validation": ["thinking_effort", "thinking_metric", "thinking_mode"],
        "selection_reason": "Fallback placeholder because selected JSON was not found. Run 12-02 first for a real selection.",
    }
    warnings.warn(f"{selected_path} fehlt. Nutze Fallback-Parameter {selected['params']} für Notebook-Aufbau/Caches.")

assert selected["model_family"] == "TabPFNClassifier"
assert selected["selection_target"] == "top10"
assert selected["selection_metric"] == "roc_auc"
assert selected["selection_validation_context"] == "X_valid, meta_year == 2023"
required_excluded = {"thinking_mode", "thinking_effort", "thinking_metric"}
if not required_excluded.issubset(set(selected.get("excluded_from_validation", []))):
    warnings.warn("Die selected JSON enthält nicht alle erwarteten Thinking-Ausschlüsse aus 12-02.")

frank_hall_validation = selected.get("frank_hall_validation", {})
validated_threshold = frank_hall_validation.get("optimized_threshold", np.nan)
validated_threshold_available = pd.notna(validated_threshold)
if validated_threshold_available:
    validated_threshold = float(validated_threshold)
else:
    warnings.warn("Kein validierter Frank-&-Hall-Threshold aus 12-02 gefunden. 12-03 optimiert nicht nach und lässt die validierte Ensemble-Klassifikation leer.")

selected_summary = pd.DataFrame([
    {"field": "model_family", "value": selected["model_family"]},
    {"field": "selection_target", "value": selected["selection_target"]},
    {"field": "selection_metric", "value": selected["selection_metric"]},
    {"field": "selection_validation_context", "value": selected["selection_validation_context"]},
    {"field": "params", "value": json.dumps(selected.get("params", {}), ensure_ascii=True, sort_keys=True)},
    {"field": "validated_threshold_available", "value": validated_threshold_available},
    {"field": "validated_threshold", "value": validated_threshold if validated_threshold_available else "missing"},
    {"field": "threshold_metric", "value": frank_hall_validation.get("threshold_metric", "missing")},
    {"field": "threshold_context", "value": frank_hall_validation.get("threshold_validation_context", "missing")},
    {"field": "selection_reason", "value": selected.get("selection_reason", "")},
])

print("=" * 88)
print("TABPFN 12-03: Geladene Auswahl und validierter Threshold aus 12-02")
print("=" * 88)
display(selected_summary)


TABPFN 12-03: Geladene Auswahl und validierter Threshold aus 12-02


,field,value
0,model_family,TabPFNClassifier
1,selection_target,top10
2,selection_metric,roc_auc
3,selection_validation_context,"X_valid, meta_year == 2023"
4,params,"{""average_before_softmax"": true, ""balance_prob..."
5,validated_threshold_available,True
6,validated_threshold,2.1
7,threshold_metric,f1
8,threshold_context,"X_valid, meta_year == 2023"
9,selection_reason,"Best validation roc_auc; average_precision, or..."


## Finale Trainings- und Testdaten laden

Das finale Training kombiniert `X_train` und `X_valid`, also alle Daten bis einschließlich 2023. Der Testsplit bleibt 2024/2025 und wird erst hier bewertet.

**Konkretes Vorgehen:** Für jedes Target wird ein finales Trainingsziel aus Train+Validation gebaut. Die drei Testtargets und die echten Ränge bleiben für Metriken, disjunkte Klassen und Case Study erhalten.

### Interpretation der finalen Datenbasis

`X_train_final` enthält Train und Validation, also alles bis einschließlich 2023. Das ist nach abgeschlossener Hyperparameterwahl erlaubt, weil 2023 nun nicht mehr zur Auswahl neuer Parameter genutzt wird. `X_test` bleibt der Zukunftstest auf 2024/2025.

In [4]:
# Features werden direkt geladen und anschließend zum finalen Trainingskontext zusammengeführt.
X_train = pd.read_pickle(MODEL_DATA_DIR / "X_train.pkl")
X_valid = pd.read_pickle(MODEL_DATA_DIR / "X_valid.pkl")
X_test = pd.read_pickle(MODEL_DATA_DIR / "X_test.pkl")
X_train_final = pd.concat([X_train, X_valid], axis=0)

# Metadaten und Gruppen definieren die stage-weite Testauswertung.
y_rank_test = pd.read_pickle(MODEL_DATA_DIR / "y_rank_test.pkl")
groups_test = pd.read_pickle(MODEL_DATA_DIR / "groups_test.pkl")
meta_test = pd.read_pickle(MODEL_DATA_DIR / "meta_test.pkl")

# Pro Target entsteht ein finales Trainingsziel und ein Testziel.
y_targets = {}
for target, cfg in TARGET_CONFIGS.items():
    y_train = pd.read_pickle(MODEL_DATA_DIR / cfg["train_file"])
    y_valid = pd.read_pickle(MODEL_DATA_DIR / cfg["valid_file"])
    y_test = pd.read_pickle(MODEL_DATA_DIR / cfg["test_file"])
    y_targets[(target, "train_final")] = pd.concat([y_train, y_valid], axis=0)
    y_targets[(target, "test")] = y_test

assert list(X_train_final.columns) == FEATURE_COLUMNS
assert list(X_test.columns) == FEATURE_COLUMNS
if X_train_final.shape != EXPECTED_SHAPES["train_final"]:
    warnings.warn(f"X_train_final Shape weicht ab: {X_train_final.shape}")
if X_test.shape != EXPECTED_SHAPES["test"]:
    warnings.warn(f"X_test Shape weicht ab: {X_test.shape}")
assert set(meta_test["meta_year"].unique()).issubset({2024, 2025})
assert meta_test["stage_id"].nunique() == 112

print("X_train_final", X_train_final.shape)
print("X_test", X_test.shape)

split_summary = pd.DataFrame([
    {"split": "train_final", "rows": len(X_train_final), "features": X_train_final.shape[1], "years": "<=2023", "role": "Fit-Kontext für finale Classifier"},
    {"split": "test_original", "rows": len(X_test), "features": X_test.shape[1], "years": ", ".join(map(str, sorted(meta_test["meta_year"].unique()))), "role": "echter Zukunftstest"},
])

target_distribution_rows = []
for target in TARGET_CONFIGS:
    for split_name, y_values in [("train_final", y_targets[(target, "train_final")]), ("test", y_targets[(target, "test")])]:
        y_series = pd.Series(y_values)
        target_distribution_rows.append({
            "target": target,
            "split": split_name,
            "rows": len(y_series),
            "positives": int(y_series.sum()),
            "positive_rate_percent": 100 * float(y_series.mean()),
        })
target_distribution_df = pd.DataFrame(target_distribution_rows)

print("=" * 88)
print("TABPFN 12-03: Finale Trainings- und Testdaten")
print("=" * 88)
display(split_summary)
display(target_distribution_df)

X_train_final (178246, 17)
X_test (17802, 17)
TABPFN 12-03: Finale Trainings- und Testdaten


,split,rows,features,years,role
0,train_final,178246,17,<=2023,Fit-Kontext für finale Classifier
1,test_original,17802,17,"2024, 2025",echter Zukunftstest


,target,split,rows,positives,positive_rate_percent
0,top5,train_final,178246,5295,2.970614
1,top5,test,17802,556,3.123245
2,top10,train_final,178246,10576,5.933373
3,top10,test,17802,1104,6.201550
4,top20,train_final,178246,21103,11.839256
5,top20,test,17802,2205,12.386249


## TabPFN-Client, finale Varianten und Thinking-Parameter definieren

Die Standardvariante verwendet exakt die in `12-02` ausgewählten normalen Classifier-Parameter. Die Thinking-Variante ist nur ein finaler Vergleich und bekommt explizit alle Thinking-relevanten API-Parameter. Zusätzlich wird ein Client-Timeout gesetzt, der größer als der serverseitige Thinking-Timeout ist.

Alte Thinking-Caches werden später nur akzeptiert, wenn der gespeicherte Fingerprint zur aktuellen Anfrage passt. Das verhindert, dass ein schneller Standardlauf versehentlich als Thinking-Lauf in die Auswertung gelangt.


In [5]:
TabPFNClassifier = None
ClientOptions = None
client_import_error = None
tabpfn_token_source = "missing"
tabpfn_token_file = PROJECT_ROOT / "Dokumentation" / "tabpfn_token.env"
if os.environ.get("TABPFN_TOKEN"):
    tabpfn_token_source = "environment"
elif tabpfn_token_file.exists():
    for line in tabpfn_token_file.read_text().splitlines():
        if line.startswith("TABPFN_TOKEN="):
            token_value = line.split("=", 1)[1].strip()
            if token_value:
                os.environ["TABPFN_TOKEN"] = token_value
                tabpfn_token_source = "Dokumentation/tabpfn_token.env"
            break

try:
    from tabpfn_client import TabPFNClassifier as ImportedTabPFNClassifier
    from tabpfn_client.client import ClientOptions as ImportedClientOptions
    try:
        from tabpfn_client import set_access_token as ImportedSetAccessToken
        if os.environ.get("TABPFN_TOKEN"):
            ImportedSetAccessToken(os.environ["TABPFN_TOKEN"])
    except Exception as exc_token:
        if os.environ.get("TABPFN_TOKEN"):
            warnings.warn(f"TABPFN_TOKEN wurde gefunden, konnte aber nicht aktiv an tabpfn_client uebergeben werden: {exc_token}")
    TabPFNClassifier = ImportedTabPFNClassifier
    ClientOptions = ImportedClientOptions
    client_source = "tabpfn_client"
except Exception as exc_client:
    try:
        from tabpfn import TabPFNClassifier as ImportedTabPFNClassifier
        TabPFNClassifier = ImportedTabPFNClassifier
        client_source = "tabpfn"
    except Exception as exc_local:
        client_import_error = f"tabpfn_client: {exc_client}; tabpfn: {exc_local}"
        client_source = None

if TabPFNClassifier is None:
    warnings.warn(f"Kein TabPFNClassifier importierbar. RUN_API-Läufe bleiben deaktiviert. {client_import_error}")
    supported_params = {}
else:
    try:
        supported_params = TabPFNClassifier().get_params()
    except Exception as exc:
        warnings.warn(f"TabPFNClassifier konnte nicht instanziiert werden: {exc}")
        supported_params = {}

FINAL_TEST_VARIANTS = [
    {
        "variant_name": "standard",
        "extra_params": {},
        "client_timeout_seconds": None,
        "role": "selected_params_without_thinking",
    },
    {
        "variant_name": "thinking_medium_roc_auc",
        "extra_params": {
            "thinking_mode": True,
            "thinking_effort": "medium",
            "thinking_metric": "roc_auc",
            "thinking_timeout_s": THINKING_TIMEOUT_SECONDS,
            "force_refit": True,
        },
        "client_timeout_seconds": THINKING_CLIENT_TIMEOUT_SECONDS,
        "role": "final_test_comparison_only",
    },
]

variant_support_rows = []
for variant in FINAL_TEST_VARIANTS:
    extra_params = variant["extra_params"]
    unsupported_params = [key for key in extra_params if key not in supported_params]
    needs_client_timeout = pd.notna(variant.get("client_timeout_seconds")) and variant.get("client_timeout_seconds") is not None
    client_timeout_supported = (ClientOptions is not None) and ("client_options" in supported_params)
    can_run = (variant["variant_name"] == "standard") or ((not unsupported_params) and ((not needs_client_timeout) or client_timeout_supported))
    variant_support_rows.append({
        "variant_name": variant["variant_name"],
        "role": variant["role"],
        "extra_params_json": json.dumps(extra_params, sort_keys=True, ensure_ascii=True, default=str),
        "client_timeout_seconds": variant.get("client_timeout_seconds"),
        "unsupported_params": ", ".join(unsupported_params),
        "client_timeout_supported": client_timeout_supported,
        "can_run": can_run,
        "client_source": client_source,
    })

variant_support = pd.DataFrame(variant_support_rows)
variant_support.to_csv(THINKING_DIR / "tabpfn_thinking_variant_support.csv", index=False)

support_summary = variant_support.groupby(["role", "can_run"], dropna=False).size().reset_index(name="variants")

print("=" * 88)
print("TABPFN 12-03: Finale Varianten und Thinking-Support")
print("=" * 88)
display(variant_support)
print("Kompakte Varianten-Zählung:")
display(support_summary)


TABPFN 12-03: Finale Varianten und Thinking-Support


,variant_name,role,extra_params_json,client_timeout_seconds,unsupported_params,client_timeout_supported,can_run,client_source
0,standard,selected_params_without_thinking,{},NaN,,True,True,tabpfn_client
1,thinking_medium_roc_auc,final_test_comparison_only,"{""force_refit"": true, ""thinking_effort"": ""medi...",3000.0,,True,True,tabpfn_client


Kompakte Varianten-Zählung:


,role,can_run,variants
0,final_test_comparison_only,True,1
1,selected_params_without_thinking,True,1


## Finale Target-Predictions laden oder erzeugen

Für jede Variante entstehen drei Prediction-Dateien: `Top5`, `Top10` und `Top20`. Erst wenn alle drei Kanäle vorhanden sind, wird eine breite finale Tabelle erzeugt.

Die Standardvariante darf vorhandene Caches weiter nutzen. Für Thinking gilt eine strengere Regel: Der Cache muss einen Runtime-Sidecar mit passendem Fingerprint, Thinking-Parametern, `force_refit`, Timeout und einer plausiblen Laufzeit enthalten. Andernfalls wird der Cache als veraltet markiert und nicht als echter Thinking-Lauf gewertet.


In [6]:
# Leseschlüssel für diese lange finale Prediction-Zelle:
# 1. Die äußere Schleife läuft über finale Varianten: Standard und optional Thinking.
# 2. Die innere Schleife läuft über die drei Targets Top5, Top10 und Top20.
# 3. Pro Target werden Request-Parameter und ein stabiler Fingerprint gebildet.
# 4. Standard-Caches bleiben offline nutzbar; Thinking-Caches müssen exakt zum Fingerprint passen.
# 5. Bei einem API-Lauf werden Fit und Prediction getrennt gemessen und im Sidecar gespeichert.
# 5a. Standardmodelle werden nach erfolgreichem Fit fuer das SHAP-Notebook 12-04 exportiert.
# 6. Erst nach vollständigen drei Target-Predictions wird die breite kombinierte Prediction-Tabelle gebaut.
# 7. Der finale Ranking-Score ist ausschließlich score_topk_raw_sum.
# 8. Keine Differenz aus kumulativen Wahrscheinlichkeiten wird als disjunkte Hauptwahrscheinlichkeit interpretiert.

runtime_rows = []
combined_predictions = []
variant_target_predictions = {}
runtime_csv_path = RESULT_DIR / "tabpfn_final_runtime_by_target.csv"
standard_model_manifest_rows = []

for variant in FINAL_TEST_VARIANTS:
    variant_name = variant["variant_name"]
    variant_label = "".join(char if char.isalnum() or char in "-_" else "_" for char in str(variant_name)).strip("_")
    variant_dir = PREDICTION_ROOT / variant_label
    variant_dir.mkdir(parents=True, exist_ok=True)
    target_predictions = {}
    support_row = variant_support[variant_support["variant_name"] == variant_name].iloc[0]
    variant_can_run = bool(support_row["can_run"])

    for target, cfg in TARGET_CONFIGS.items():
        print(f"Variant={variant_name}, target={target}")
        pred_path = variant_dir / f"tabpfn_final_predictions_{target}_2024_2025.csv"
        sidecar_path = pred_path.with_name(f"{pred_path.stem}_runtime.json")
        model_artifact_path = STANDARD_MODEL_DIR / f"tabpfn_standard_{target}_model.pkl"
        run_start = perf_counter()

        selected_params = selected.get("params", {})
        candidate_params = {**selected_params, **variant["extra_params"]}
        params = {key: value for key, value in candidate_params.items() if key in supported_params}
        if client_source == "tabpfn_client" and "n_estimators" in params and int(params["n_estimators"]) > TABPFN_CLIENT_N_ESTIMATORS_MAX:
            warnings.warn(f"n_estimators={params['n_estimators']} überschreitet das API-Limit; setze auf {TABPFN_CLIENT_N_ESTIMATORS_MAX}.")
            params["n_estimators"] = TABPFN_CLIENT_N_ESTIMATORS_MAX

        client_timeout_seconds = variant.get("client_timeout_seconds")
        constructor_params = params.copy()
        if client_timeout_seconds is not None and ClientOptions is not None and "client_options" in supported_params:
            constructor_params["client_options"] = ClientOptions(timeout=float(client_timeout_seconds))

        params_for_records = {key: value for key, value in params.items() if key != "client_options"}
        request_payload = {
            "variant_name": variant_name,
            "target": target,
            "params": params_for_records,
            "client_timeout_seconds": client_timeout_seconds,
            "train_rows": len(X_train_final),
            "test_rows": len(X_test),
            "feature_columns": FEATURE_COLUMNS,
            "validated_threshold": validated_threshold if validated_threshold_available else None,
        }
        request_payload_json = json.dumps(request_payload, sort_keys=True, ensure_ascii=True, default=str)
        request_fingerprint = hashlib.sha256(request_payload_json.encode("utf-8")).hexdigest()
        thinking_requested = bool(params_for_records.get("thinking_mode", False) or params_for_records.get("thinking_effort"))

        runtime = {
            "variant_name": variant_name,
            "target": target,
            "runtime_status": None,
            "fit_seconds": np.nan,
            "predict_seconds": np.nan,
            "total_seconds": np.nan,
            "cache_load_seconds": np.nan,
            "runtime_time_source": "not_available",
            "api_run_timestamp": None,
            "runtime_complete_for_comparison": False,
            "train_rows": len(X_train_final),
            "test_rows": len(X_test),
            "prediction_path": str(pred_path),
            "params_json": json.dumps(params_for_records, sort_keys=True, ensure_ascii=True, default=str),
            "request_fingerprint": request_fingerprint,
            "client_timeout_seconds": client_timeout_seconds,
            "thinking_requested": thinking_requested,
            "thinking_effort": params_for_records.get("thinking_effort"),
            "thinking_metric": params_for_records.get("thinking_metric"),
            "thinking_timeout_s": params_for_records.get("thinking_timeout_s"),
            "force_refit": params_for_records.get("force_refit", False),
            "thinking_verified_by_runtime": not thinking_requested,
            "error_message": None,
        }

        pred = None
        cache_allowed = pred_path.exists() and (not FORCE_RERUN) and (variant_name not in FORCE_RERUN_VARIANTS) and variant_can_run
        if cache_allowed:
            sidecar_record = {}
            if sidecar_path.exists():
                with open(sidecar_path) as f:
                    sidecar_record = json.load(f)

            fingerprint_matches = sidecar_record.get("request_fingerprint") == request_fingerprint
            sidecar_says_thinking = bool(sidecar_record.get("thinking_requested", False))
            sidecar_verified = bool(sidecar_record.get("thinking_verified_by_runtime", False))

            if thinking_requested and not (fingerprint_matches and sidecar_says_thinking and sidecar_verified):
                runtime["runtime_status"] = "cache_stale_thinking_fingerprint"
                runtime["runtime_time_source"] = "stale_cache_rejected"
                runtime["total_seconds"] = round(perf_counter() - run_start, 3)
                runtime["error_message"] = "Thinking-Cache ohne passenden Fingerprint oder ohne bestätigten Thinking-Sidecar wurde verworfen."
            else:
                cache_start = perf_counter()
                pred = pd.read_csv(pred_path)
                runtime["cache_load_seconds"] = round(perf_counter() - cache_start, 3)

                if variant_name == "standard":
                    model_artifact_exists = model_artifact_path.exists()
                    model_export_status = "existing_model_artifact" if model_artifact_exists else "missing_model_artifact_cache_only"
                    model_export_error = None
                    if not model_artifact_exists:
                        model_export_error = (
                            "Prediction-Cache wurde genutzt, aber kein gespeichertes fitted Standardmodell gefunden. "
                            "Fuer SHAP muss 12-03 mit einem echten Standard-Fit und Modell-Export neu ausgefuehrt werden."
                        )
                        warnings.warn(f"{variant_name}/{target}: {model_export_error}")
                    standard_model_manifest_rows.append({
                        "variant_name": variant_name,
                        "target": target,
                        "model_family": selected.get("model_family", "TabPFNClassifier"),
                        "model_path": str(model_artifact_path),
                        "model_file_exists": bool(model_artifact_exists),
                        "export_status": model_export_status,
                        "export_error": model_export_error,
                        "source": "prediction_cache",
                        "params": params_for_records,
                        "feature_columns": FEATURE_COLUMNS,
                        "train_shape": list(X_train_final.shape),
                        "test_shape": list(X_test.shape),
                        "prediction_path": str(pred_path),
                        "request_fingerprint": request_fingerprint,
                        "created_at_utc": datetime.now(timezone.utc).isoformat(),
                    })

                previous_runtime = None
                if sidecar_record:
                    has_fit = pd.notna(sidecar_record.get("fit_seconds"))
                    has_predict = pd.notna(sidecar_record.get("predict_seconds"))
                    has_total = pd.notna(sidecar_record.get("total_seconds"))
                    if has_fit and has_predict and has_total:
                        previous_runtime = sidecar_record
                        previous_runtime["runtime_time_source"] = sidecar_record.get("runtime_time_source", "runtime_sidecar")

                if previous_runtime is None and runtime_csv_path.exists() and not thinking_requested:
                    previous_runtime_df = pd.read_csv(runtime_csv_path)
                    required_cols = {"variant_name", "target"}
                    if required_cols.issubset(previous_runtime_df.columns):
                        matches = previous_runtime_df[
                            (previous_runtime_df["variant_name"].astype(str) == str(variant_name))
                            & (previous_runtime_df["target"].astype(str) == str(target))
                        ].copy()
                        for _, previous_row in matches.iterrows():
                            previous_record = previous_row.to_dict()
                            has_fit = pd.notna(previous_record.get("fit_seconds"))
                            has_predict = pd.notna(previous_record.get("predict_seconds"))
                            has_total = pd.notna(previous_record.get("total_seconds"))
                            if has_fit and has_predict and has_total:
                                previous_runtime = previous_record
                                previous_runtime["runtime_time_source"] = "runtime_csv"
                                break

                if previous_runtime is None:
                    warnings.warn(f"Cache-Hit für {variant_name}/{target}, aber keine frühere API-Zeit gefunden. Laufzeitvergleich wird ausgeschlossen.")
                    runtime["runtime_status"] = "cache_hit_missing_api_runtime"
                    runtime["runtime_time_source"] = "missing_api_runtime"
                else:
                    runtime["fit_seconds"] = previous_runtime.get("fit_seconds")
                    runtime["predict_seconds"] = previous_runtime.get("predict_seconds")
                    runtime["total_seconds"] = previous_runtime.get("total_seconds")
                    runtime["runtime_status"] = "cache_hit"
                    runtime["runtime_time_source"] = previous_runtime.get("runtime_time_source", "runtime_sidecar")
                    runtime["api_run_timestamp"] = previous_runtime.get("api_run_timestamp")
                    runtime["thinking_verified_by_runtime"] = bool(previous_runtime.get("thinking_verified_by_runtime", not thinking_requested))
                    runtime["runtime_complete_for_comparison"] = bool(previous_runtime.get("runtime_complete_for_comparison", False))

                sidecar_payload = {key: runtime.get(key) for key in [
                    "variant_name", "target", "fit_seconds", "predict_seconds", "total_seconds",
                    "runtime_status", "runtime_time_source", "api_run_timestamp", "runtime_complete_for_comparison",
                    "train_rows", "test_rows", "params_json", "request_fingerprint", "client_timeout_seconds",
                    "thinking_requested", "thinking_effort", "thinking_metric", "thinking_timeout_s", "force_refit",
                    "thinking_verified_by_runtime", "prediction_path",
                ]}
                with open(sidecar_path, "w") as f:
                    json.dump(sidecar_payload, f, indent=2, ensure_ascii=True, default=str)

        if pred is None:
            if not variant_can_run:
                runtime["runtime_status"] = "unsupported_variant_params"
                runtime["error_message"] = f"Nicht unterstützte oder unvollständige Variant-Parameter: {support_row.get('unsupported_params', '')}"
                runtime["total_seconds"] = round(perf_counter() - run_start, 3)

            elif not RUN_API:
                if runtime["runtime_status"] is None:
                    runtime["runtime_status"] = "skipped_missing_cache"
                    runtime["total_seconds"] = round(perf_counter() - run_start, 3)

            elif TabPFNClassifier is None:
                runtime["runtime_status"] = "missing_classifier"
                runtime["error_message"] = client_import_error
                runtime["total_seconds"] = round(perf_counter() - run_start, 3)

            else:
                try:
                    api_timestamp = datetime.now(timezone.utc).isoformat()

                    fit_start = perf_counter()
                    model = TabPFNClassifier(**constructor_params)
                    model.fit(X_train_final, y_targets[(target, "train_final")])
                    runtime["fit_seconds"] = round(perf_counter() - fit_start, 3)

                    predict_start = perf_counter()
                    proba = model.predict_proba(X_test)
                    classes = list(model.classes_)
                    if 1 not in classes:
                        raise ValueError(f"Positive Klasse 1 fehlt in model.classes_: {classes}")
                    positive_scores = proba[:, classes.index(1)]
                    runtime["predict_seconds"] = round(perf_counter() - predict_start, 3)

                    pred = meta_test.reset_index(drop=True).copy()
                    pred["stage_id"] = pd.Series(groups_test).reset_index(drop=True).astype(str).values
                    pred["actual_rank"] = pd.Series(y_rank_test).reset_index(drop=True).values
                    pred[cfg["actual_col"]] = pd.Series(y_targets[(target, "test")]).reset_index(drop=True).astype(int).values
                    pred[cfg["score_col"]] = positive_scores
                    pred["target"] = target
                    pred["variant_name"] = variant_name
                    pred["params_json"] = json.dumps(params_for_records, sort_keys=True, ensure_ascii=True, default=str)
                    pred["request_fingerprint"] = request_fingerprint
                    for key, value in params_for_records.items():
                        pred[key] = value
                    pred.to_csv(pred_path, index=False)

                    if variant_name == "standard":
                        model_export_status = "exported"
                        model_export_error = None
                        try:
                            joblib.dump(model, model_artifact_path)
                        except Exception as exc:
                            model_export_status = "export_failed_not_serializable"
                            model_export_error = str(exc)
                            warnings.warn(f"{variant_name}/{target}: Standardmodell konnte nicht fuer SHAP serialisiert werden: {exc}")
                        standard_model_manifest_rows.append({
                            "variant_name": variant_name,
                            "target": target,
                            "model_family": selected.get("model_family", "TabPFNClassifier"),
                            "model_path": str(model_artifact_path),
                            "model_file_exists": bool(model_artifact_path.exists()),
                            "export_status": model_export_status,
                            "export_error": model_export_error,
                            "source": "current_api_fit",
                            "params": params_for_records,
                            "feature_columns": FEATURE_COLUMNS,
                            "train_shape": list(X_train_final.shape),
                            "test_shape": list(X_test.shape),
                            "prediction_path": str(pred_path),
                            "request_fingerprint": request_fingerprint,
                            "created_at_utc": datetime.now(timezone.utc).isoformat(),
                        })

                    runtime["total_seconds"] = round(perf_counter() - run_start, 3)
                    runtime["runtime_status"] = "api_run"
                    runtime["runtime_time_source"] = "current_api_run"
                    runtime["api_run_timestamp"] = api_timestamp
                    runtime["thinking_verified_by_runtime"] = (runtime["fit_seconds"] >= THINKING_MIN_FIT_SECONDS_FOR_WARNING) if thinking_requested else True
                    runtime["runtime_complete_for_comparison"] = bool(runtime["thinking_verified_by_runtime"])
                    if thinking_requested and not runtime["thinking_verified_by_runtime"]:
                        runtime["runtime_status"] = "api_run_thinking_unverified_short_fit"
                        warnings.warn(f"{variant_name}/{target}: Thinking-Lauf war mit {runtime['fit_seconds']} Sekunden ungewöhnlich kurz und wird nicht als echter Thinking-Vergleich gewertet.")

                    sidecar_payload = {key: runtime.get(key) for key in [
                        "variant_name", "target", "fit_seconds", "predict_seconds", "total_seconds",
                        "runtime_status", "runtime_time_source", "api_run_timestamp", "runtime_complete_for_comparison",
                        "train_rows", "test_rows", "params_json", "request_fingerprint", "client_timeout_seconds",
                        "thinking_requested", "thinking_effort", "thinking_metric", "thinking_timeout_s", "force_refit",
                        "thinking_verified_by_runtime", "prediction_path",
                    ]}
                    sidecar_payload["request_payload_json"] = request_payload_json
                    with open(sidecar_path, "w") as f:
                        json.dump(sidecar_payload, f, indent=2, ensure_ascii=True, default=str)

                except Exception as exc:
                    runtime["runtime_status"] = "error"
                    runtime["error_message"] = str(exc)
                    runtime["total_seconds"] = round(perf_counter() - run_start, 3)

        runtime_rows.append(runtime)
        if pred is not None:
            target_predictions[target] = pred
            variant_target_predictions[(variant_name, target)] = pred

    # Erst wenn alle drei Target-Dateien vorhanden sind, wird die breite finale Prediction-Tabelle gebaut.
    if set(target_predictions) == set(TARGET_CONFIGS):
        combined = target_predictions["top5"].copy()
        if "target" in combined.columns:
            combined = combined.drop(columns=["target"])
        combined["actual_top10"] = target_predictions["top10"]["actual_top10"].reset_index(drop=True).astype(int)
        combined["actual_top20"] = target_predictions["top20"]["actual_top20"].reset_index(drop=True).astype(int)
        combined["p_top10_raw"] = target_predictions["top10"]["p_top10_raw"].reset_index(drop=True).astype(float)
        combined["p_top20_raw"] = target_predictions["top20"]["p_top20_raw"].reset_index(drop=True).astype(float)

        # Die abgestufte Relevanz wird nur für Rankingmetriken genutzt.
        numeric_rank = pd.to_numeric(combined["actual_rank"], errors="coerce")
        combined["actual_relevance_graded"] = np.select(
            [
                numeric_rank.between(1, 5, inclusive="both"),
                numeric_rank.between(6, 10, inclusive="both"),
                numeric_rank.between(11, 20, inclusive="both"),
            ],
            [3, 2, 1],
            default=0,
        ).astype(int)

        # Dies ist der einzige finale Ranking-Score.
        combined["score_topk_raw_sum"] = combined["p_top5_raw"] + combined["p_top10_raw"] + combined["p_top20_raw"]
        combined["rank_topk_raw_sum"] = combined.groupby("stage_id")["score_topk_raw_sum"].rank(method="first", ascending=False).astype(int)
        combined["score_name"] = "score_topk_raw_sum"

        combined["predicted_rank_bucket"] = np.select(
            [
                combined["rank_topk_raw_sum"] <= 5,
                combined["rank_topk_raw_sum"] <= 10,
                combined["rank_topk_raw_sum"] <= 20,
            ],
            ["top5", "rank_6_10", "rank_11_20"],
            default="outside_top20",
        )
        combined["rank_forced_ordinal_class"] = np.select(
            [
                combined["rank_topk_raw_sum"] <= 5,
                combined["rank_topk_raw_sum"] <= 10,
                combined["rank_topk_raw_sum"] <= 20,
            ],
            [3, 2, 1],
            default=0,
        ).astype(int)

        combined["violates_top5_le_top10"] = combined["p_top5_raw"] > combined["p_top10_raw"]
        combined["violates_top10_le_top20"] = combined["p_top10_raw"] > combined["p_top20_raw"]
        combined["is_cumulative_probability_monotonic"] = ~(combined["violates_top5_le_top10"] | combined["violates_top10_le_top20"])
        combined["has_monotonicity_violation"] = ~combined["is_cumulative_probability_monotonic"]

        combined["fh_seq_class_0_5"] = 0
        seq_top20 = combined["p_top20_raw"] >= 0.5
        seq_top10 = seq_top20 & (combined["p_top10_raw"] >= 0.5)
        seq_top5 = seq_top10 & (combined["p_top5_raw"] >= 0.5)
        combined.loc[seq_top20, "fh_seq_class_0_5"] = 1
        combined.loc[seq_top10, "fh_seq_class_0_5"] = 2
        combined.loc[seq_top5, "fh_seq_class_0_5"] = 3
        fh_label_map = {0: "outside_top20", 1: "rank_11_20", 2: "rank_6_10", 3: "top5"}
        combined["fh_seq_class_label_0_5"] = combined["fh_seq_class_0_5"].map(fh_label_map)

        combined["ensemble_pred_top10_native_0_5"] = (combined["score_topk_raw_sum"] >= 0.5).astype(int)
        if validated_threshold_available:
            combined["ensemble_pred_top10_validated_threshold"] = (combined["score_topk_raw_sum"] >= validated_threshold).astype(int)
            combined["validated_top10_threshold"] = validated_threshold
        else:
            combined["ensemble_pred_top10_validated_threshold"] = np.nan
            combined["validated_top10_threshold"] = np.nan

        combined.to_csv(variant_dir / "tabpfn_final_predictions_2024_2025_all_targets.csv", index=False)
        combined_predictions.append(combined)

standard_model_targets_with_files = {
    row["target"]
    for row in standard_model_manifest_rows
    if row.get("variant_name") == "standard" and row.get("model_file_exists") and row.get("export_status") in {"exported", "existing_model_artifact"}
}
standard_model_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_dir": str(STANDARD_MODEL_DIR),
    "required_targets": list(TARGET_CONFIGS.keys()),
    "feature_columns": FEATURE_COLUMNS,
    "all_standard_models_available": set(TARGET_CONFIGS.keys()).issubset(standard_model_targets_with_files),
    "rows": standard_model_manifest_rows,
}
with open(STANDARD_MODEL_MANIFEST_PATH, "w") as f:
    json.dump(standard_model_manifest, f, indent=2, ensure_ascii=True, default=str)
standard_model_manifest_df = pd.DataFrame(standard_model_manifest_rows)

runtime_df = pd.DataFrame(runtime_rows)
if not runtime_df.empty:
    runtime_df["runtime_complete_for_comparison"] = runtime_df["runtime_complete_for_comparison"].fillna(False).astype(bool)
    runtime_df["thinking_verified_by_runtime"] = runtime_df["thinking_verified_by_runtime"].fillna(False).astype(bool)
else:
    runtime_df = pd.DataFrame(columns=["variant_name", "target", "runtime_complete_for_comparison", "thinking_verified_by_runtime"])

runtime_df.to_csv(RESULT_DIR / "tabpfn_final_runtime_by_target.csv", index=False)
runtime_df.to_csv(RESULT_DIR / "tabpfn_final_runtime.csv", index=False)

if runtime_df.empty:
    runtime_status_summary = pd.DataFrame(columns=["variant_name", "runtime_status", "targets"])
else:
    runtime_status_summary = runtime_df.groupby(["variant_name", "runtime_status"], dropna=False).size().reset_index(name="targets")

combined_sanity_rows = []
for combined in combined_predictions:
    combined_sanity_rows.append({
        "variant_name": combined["variant_name"].iloc[0],
        "rows": len(combined),
        "stages": combined["stage_id"].nunique(),
        "monotonicity_violation_rate": float(combined["has_monotonicity_violation"].mean()),
        "top5_gt_top10_rate": float(combined["violates_top5_le_top10"].mean()),
        "top10_gt_top20_rate": float(combined["violates_top10_le_top20"].mean()),
        "score_min": float(combined["score_topk_raw_sum"].min()),
        "score_mean": float(combined["score_topk_raw_sum"].mean()),
        "score_max": float(combined["score_topk_raw_sum"].max()),
    })
combined_sanity_df = pd.DataFrame(combined_sanity_rows)

print("=" * 88)
print("TABPFN 12-03: Target-Prediction-Status und kombinierte Score-Prüfung")
print("=" * 88)
display(runtime_status_summary)
display(combined_sanity_df)
print("Standardmodell-Exportstatus fuer 12-04 SHAP:")
if standard_model_manifest_df.empty:
    display(pd.DataFrame(columns=["variant_name", "target", "model_path", "export_status", "model_file_exists"]))
else:
    display(standard_model_manifest_df[["variant_name", "target", "model_path", "export_status", "model_file_exists", "source", "export_error"]])


Variant=standard, target=top5
00:00 Fitting... -

The provided train set hashes match previously uploaded train sets.


00:05 Fitting... Done!
00:00 Predicting... -

The provided test set hash matches a previously uploaded test set.


00:28 Predicting... Done!
Variant=standard, target=top10
00:00 Fitting... -

...

00:27 Predicting... Done!
Variant=thinking_medium_roc_auc, target=top5
Variant=thinking_medium_roc_auc, target=top10
Variant=thinking_medium_roc_auc, target=top20
TABPFN 12-03: Target-Prediction-Status und kombinierte Score-Prüfung


,variant_name,runtime_status,targets
0,standard,api_run,3
1,thinking_medium_roc_auc,cache_hit,3


,variant_name,rows,stages,monotonicity_violation_rate,top5_gt_top10_rate,top10_gt_top20_rate,score_min,score_mean,score_max
0,standard,17802,112,0.533704,0.341366,0.323615,0.007444,0.791236,2.968350
1,thinking_medium_roc_auc,17802,112,0.486799,0.362712,0.241939,0.009619,0.816401,2.964245


Standardmodell-Exportstatus fuer 12-04 SHAP:


,variant_name,target,model_path,export_status,model_file_exists,source,export_error
0,standard,top5,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,exported,True,current_api_fit,None
1,standard,top10,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,exported,True,current_api_fit,None
2,standard,top20,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,exported,True,current_api_fit,None


## Laufzeiten pro Variante zusammenfassen

Die Laufzeitbewertung bleibt getrennt von der Metrikbewertung. Eine Variante gilt nur dann als vollständig vergleichbar, wenn alle drei Targets vorhanden sind und für alle drei Targets ursprüngliche API-Zeiten vorliegen.

### Warum unvollständige Varianten sichtbar bleiben

Eine Variante mit fehlendem Target kann Metriken nicht vollständig liefern. Trotzdem ist ihr Status wichtig, weil er erklärt, ob ein Vergleich an fehlenden Caches, fehlender API-Zeit oder nicht unterstützten Thinking-Parametern scheitert. Diese Transparenz verhindert, dass nur erfolgreiche Läufe sichtbar bleiben.

In [7]:
runtime_by_variant_rows = []
for variant_name, part in runtime_df.groupby("variant_name"):
    completed_statuses = part["runtime_status"].isin(["api_run", "cache_hit", "cache_hit_missing_api_runtime", "api_run_thinking_unverified_short_fit"])
    runtime_complete = part["runtime_complete_for_comparison"].fillna(False).astype(bool)
    comparable_part = part[runtime_complete].copy()
    thinking_requested_part = part["thinking_requested"].fillna(False).astype(bool) if "thinking_requested" in part.columns else pd.Series(False, index=part.index)
    thinking_verified_part = part["thinking_verified_by_runtime"].fillna(False).astype(bool) if "thinking_verified_by_runtime" in part.columns else pd.Series(True, index=part.index)
    thinking_targets_verified = int((thinking_verified_part | ~thinking_requested_part).sum())

    runtime_by_variant_rows.append({
        "variant_name": variant_name,
        "targets_completed": int(completed_statuses.sum()),
        "targets_with_api_runtime": int(runtime_complete.sum()),
        "targets_expected": len(TARGET_CONFIGS),
        "thinking_targets_verified": thinking_targets_verified,
        "runtime_status": ", ".join(sorted(set(part["runtime_status"].astype(str)))),
        "total_fit_seconds": comparable_part["fit_seconds"].astype(float).sum() if not comparable_part.empty else np.nan,
        "total_predict_seconds": comparable_part["predict_seconds"].astype(float).sum() if not comparable_part.empty else np.nan,
        "total_runtime_seconds": comparable_part["total_seconds"].astype(float).sum() if not comparable_part.empty else np.nan,
        "mean_runtime_seconds_per_target": comparable_part["total_seconds"].astype(float).mean() if not comparable_part.empty else np.nan,
        "runtime_complete_for_comparison": bool((completed_statuses.sum() == len(TARGET_CONFIGS)) and (runtime_complete.sum() == len(TARGET_CONFIGS)) and (thinking_targets_verified == len(TARGET_CONFIGS))),
    })

runtime_by_variant = pd.DataFrame(runtime_by_variant_rows)
runtime_by_variant.to_csv(RESULT_DIR / "tabpfn_final_runtime_by_variant.csv", index=False)
runtime_by_variant.to_csv(THINKING_DIR / "tabpfn_thinking_runtime.csv", index=False)

print("=" * 88)
print("TABPFN 12-03: Laufzeitvergleich pro Variante")
print("=" * 88)
display(runtime_by_variant)


TABPFN 12-03: Laufzeitvergleich pro Variante


,variant_name,targets_completed,targets_with_api_runtime,targets_expected,thinking_targets_verified,runtime_status,total_fit_seconds,total_predict_seconds,total_runtime_seconds,mean_runtime_seconds_per_target,runtime_complete_for_comparison
0,standard,3,3,3,3,api_run,17.043,94.931,112.389,37.463,True
1,thinking_medium_roc_auc,3,3,3,3,cache_hit,270.028,89.884,360.333,120.111,True


## Finale Klassifikations- und Rankingmetriken berechnen

Diese Zelle trennt bewusst zwei Ebenen. Erstens werden die binären Top-K-Entscheidungen mit festen Thresholds bewertet. Dazu gehören die nativen Einzelkanäle bei `0.5`, die naive Ensemble-Baseline bei `0.5` und, falls vorhanden, der in `12-02` validierte Ensemble-Threshold. Zweitens wird das stage-weite Ranking nach `score_topk_raw_sum` bewertet.

Die Klassenverteilungen nutzen keine Differenzen wie `p_top10_raw - p_top5_raw`. Stattdessen werden der tatsächliche Ranking-Bucket und die sequenzielle Frank-&-Hall-Klasse mit Threshold `0.5` dokumentiert.


In [8]:
# Leseschlüssel für die Metrikzelle:
# 1. Zuerst werden Monotonie-Diagnostics und Klassenverteilungen dokumentiert.
# 2. Danach werden binäre Klassifikationsmetriken und Konfusionsmatrizen für feste Thresholds erzeugt.
# 3. Zusätzlich wird eine 4x4-Matrix für die sequenzielle Frank-&-Hall-Klasse geschrieben.
# 4. Anschließend wird jede Etappe separat für die Rankingmetriken ausgewertet.
# 5. NDCG nutzt abgestufte Relevanz: Top5 zählt stärker als Rang 6-10 und Rang 11-20.
# 6. Winner-Hit-Rates prüfen, ob der echte Sieger in den vorhergesagten Top-n enthalten ist.
# 7. MAP betrachtet alle echten Top20-Fahrer als relevante Menge.

stage_metric_rows = []
class_distribution_rows = []
monotonicity_rows = []
classification_metric_rows = []
confusion_matrix_rows = []
ordinal_confusion_rows = []

for combined in combined_predictions:
    variant_name = combined["variant_name"].iloc[0]

    for scope_name, scope_part in [("overall", combined), *[(f"year_{year}", year_part) for year, year_part in combined.groupby("meta_year")]]:
        monotonicity_rows.append({
            "variant_name": variant_name,
            "scope": scope_name,
            "rows": int(len(scope_part)),
            "monotonic_rows": int(scope_part["is_cumulative_probability_monotonic"].sum()),
            "monotonicity_violation_rows": int(scope_part["has_monotonicity_violation"].sum()),
            "monotonicity_violation_rate": float(scope_part["has_monotonicity_violation"].mean()),
            "top5_gt_top10_rate": float(scope_part["violates_top5_le_top10"].mean()),
            "top10_gt_top20_rate": float(scope_part["violates_top10_le_top20"].mean()),
        })

        for distribution_type, label_col in [("predicted_rank_bucket", "predicted_rank_bucket"), ("fh_seq_class_0_5", "fh_seq_class_label_0_5")]:
            distribution = scope_part.groupby(label_col, dropna=False).size().reset_index(name="rows")
            for _, distribution_row in distribution.iterrows():
                class_distribution_rows.append({
                    "variant_name": variant_name,
                    "scope": scope_name,
                    "distribution_type": distribution_type,
                    "class_label": distribution_row[label_col],
                    "rows": int(distribution_row["rows"]),
                    "row_share": float(distribution_row["rows"] / len(scope_part)) if len(scope_part) else np.nan,
                })

        evaluation_specs = []
        for target, cfg in TARGET_CONFIGS.items():
            evaluation_specs.append({
                "evaluation_name": f"{target}_native_0_5",
                "target": target,
                "actual_col": cfg["actual_col"],
                "score_col": cfg["score_col"],
                "score_name": cfg["score_col"],
                "threshold": 0.5,
                "threshold_source": "native_0_5",
            })
        evaluation_specs.append({
            "evaluation_name": "frank_hall_top10_native_0_5",
            "target": "top10",
            "actual_col": "actual_top10",
            "score_col": "score_topk_raw_sum",
            "score_name": "score_topk_raw_sum",
            "threshold": 0.5,
            "threshold_source": "native_0_5",
        })
        if validated_threshold_available:
            evaluation_specs.append({
                "evaluation_name": "frank_hall_top10_validated_threshold",
                "target": "top10",
                "actual_col": "actual_top10",
                "score_col": "score_topk_raw_sum",
                "score_name": "score_topk_raw_sum",
                "threshold": validated_threshold,
                "threshold_source": "optimized_valid_2023_f1",
            })

        for spec in evaluation_specs:
            y_true = scope_part[spec["actual_col"]].astype(int)
            y_score = scope_part[spec["score_col"]].astype(float)
            y_pred = (y_score >= float(spec["threshold"])).astype(int)
            matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
            roc_auc_value = roc_auc_score(y_true, y_score) if y_true.nunique() == 2 else np.nan
            average_precision_value = average_precision_score(y_true, y_score) if y_true.nunique() == 2 else np.nan

            classification_metric_rows.append({
                "variant_name": variant_name,
                "scope": scope_name,
                "evaluation_name": spec["evaluation_name"],
                "target": spec["target"],
                "actual_col": spec["actual_col"],
                "score_name": spec["score_name"],
                "threshold": float(spec["threshold"]),
                "threshold_source": spec["threshold_source"],
                "rows": int(len(y_true)),
                "actual_positives": int(y_true.sum()),
                "predicted_positives": int(y_pred.sum()),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "roc_auc": roc_auc_value,
                "average_precision": average_precision_value,
            })

            for actual_label, row_values in zip([0, 1], matrix):
                for predicted_label, count in zip([0, 1], row_values):
                    confusion_matrix_rows.append({
                        "variant_name": variant_name,
                        "scope": scope_name,
                        "evaluation_name": spec["evaluation_name"],
                        "target": spec["target"],
                        "threshold": float(spec["threshold"]),
                        "threshold_source": spec["threshold_source"],
                        "actual_label": int(actual_label),
                        "predicted_label": int(predicted_label),
                        "count": int(count),
                    })

        ordinal_matrix = confusion_matrix(scope_part["actual_relevance_graded"].astype(int), scope_part["fh_seq_class_0_5"].astype(int), labels=[0, 1, 2, 3])
        ordinal_label_map = {0: "outside_top20", 1: "rank_11_20", 2: "rank_6_10", 3: "top5"}
        for actual_label, row_values in zip([0, 1, 2, 3], ordinal_matrix):
            for predicted_label, count in zip([0, 1, 2, 3], row_values):
                ordinal_confusion_rows.append({
                    "variant_name": variant_name,
                    "scope": scope_name,
                    "threshold_source": "sequential_native_0_5",
                    "actual_class": int(actual_label),
                    "actual_class_label": ordinal_label_map[actual_label],
                    "predicted_class": int(predicted_label),
                    "predicted_class_label": ordinal_label_map[predicted_label],
                    "count": int(count),
                })

    for stage_id, stage in combined.groupby("stage_id"):
        ordered = stage.sort_values("rank_topk_raw_sum", ascending=True).copy()
        ideal = stage.sort_values("actual_relevance_graded", ascending=False).copy()
        row = {
            "variant_name": variant_name,
            "score_name": "score_topk_raw_sum",
            "stage_id": stage_id,
            "meta_year": stage["meta_year"].iloc[0],
            "meta_race": stage["meta_race"].iloc[0],
            "stage_nr": stage["stage_nr"].iloc[0],
            "rows": len(stage),
        }

        # NDCG bewertet, ob echte Top5/Top10/Top20-Fahrer weit oben im vorhergesagten Ranking stehen.
        for k in [5, 10, 20]:
            predicted_relevance = ordered["actual_relevance_graded"].astype(float).to_numpy()[:k]
            ideal_relevance = ideal["actual_relevance_graded"].astype(float).to_numpy()[:k]
            predicted_discounts = np.log2(np.arange(2, len(predicted_relevance) + 2))
            ideal_discounts = np.log2(np.arange(2, len(ideal_relevance) + 2))
            dcg = float(np.sum((2 ** predicted_relevance - 1) / predicted_discounts)) if len(predicted_relevance) else 0.0
            idcg = float(np.sum((2 ** ideal_relevance - 1) / ideal_discounts)) if len(ideal_relevance) else 0.0
            row[f"ndcg_at_{k}"] = dcg / idcg if idcg > 0 else np.nan

        # Winner-Hit zeigt, ob der echte Etappensieger im vorhergesagten Top-n-Fenster auftaucht.
        winner_rows = stage[pd.to_numeric(stage["actual_rank"], errors="coerce") == 1]
        if winner_rows.empty:
            for n in [1, 5, 10, 20]:
                row[f"top_{n}_accuracy"] = np.nan
        else:
            winner_predicted_rank = float(winner_rows["rank_topk_raw_sum"].min())
            for n in [1, 5, 10, 20]:
                row[f"top_{n}_accuracy"] = float(winner_predicted_rank <= n)

        # MAP nutzt alle echten Top20-Fahrer als relevante Menge.
        relevant = ordered["actual_top20"].astype(int).to_numpy()
        relevant_total = int(relevant.sum())
        if relevant_total == 0:
            row["map"] = np.nan
        else:
            cumulative_hits = np.cumsum(relevant)
            ranks = np.arange(1, len(relevant) + 1)
            precision_at_rank = cumulative_hits / ranks
            row["map"] = float(np.sum(precision_at_rank * relevant) / relevant_total)

        stage_metric_rows.append(row)

stage_metric_columns = [
    "variant_name", "score_name", "stage_id", "meta_year", "meta_race", "stage_nr", "rows",
    "ndcg_at_5", "ndcg_at_10", "ndcg_at_20",
    "top_1_accuracy", "top_5_accuracy", "top_10_accuracy", "top_20_accuracy", "map",
]
class_distribution_columns = ["variant_name", "scope", "distribution_type", "class_label", "rows", "row_share"]
monotonicity_columns = [
    "variant_name", "scope", "rows", "monotonic_rows", "monotonicity_violation_rows",
    "monotonicity_violation_rate", "top5_gt_top10_rate", "top10_gt_top20_rate",
]
classification_metric_columns = [
    "variant_name", "scope", "evaluation_name", "target", "actual_col", "score_name",
    "threshold", "threshold_source", "rows", "actual_positives", "predicted_positives",
    "precision", "recall", "f1", "roc_auc", "average_precision",
]
confusion_matrix_columns = [
    "variant_name", "scope", "evaluation_name", "target", "threshold", "threshold_source",
    "actual_label", "predicted_label", "count",
]
ordinal_confusion_columns = [
    "variant_name", "scope", "threshold_source", "actual_class", "actual_class_label",
    "predicted_class", "predicted_class_label", "count",
]
stage_metrics_df = pd.DataFrame(stage_metric_rows, columns=stage_metric_columns)
class_distribution_df = pd.DataFrame(class_distribution_rows, columns=class_distribution_columns)
monotonicity_df = pd.DataFrame(monotonicity_rows, columns=monotonicity_columns)
classification_metrics_df = pd.DataFrame(classification_metric_rows, columns=classification_metric_columns)
confusion_matrices_df = pd.DataFrame(confusion_matrix_rows, columns=confusion_matrix_columns)
ordinal_confusion_matrix_df = pd.DataFrame(ordinal_confusion_rows, columns=ordinal_confusion_columns)

summary_metric_cols = [
    "ndcg_at_5", "ndcg_at_10", "ndcg_at_20",
    "top_1_accuracy", "top_5_accuracy", "top_10_accuracy", "top_20_accuracy",
    "map",
]

overall_rows = []
by_year_rows = []
if not stage_metrics_df.empty:
    for variant_name, part in stage_metrics_df.groupby("variant_name"):
        overall_row = {
            "variant_name": variant_name,
            "score_name": "score_topk_raw_sum",
            "test_scope": "overall",
            "n_stages": part["stage_id"].nunique(),
        }
        for col in summary_metric_cols:
            overall_row[col] = part[col].astype(float).mean()
        overall_rows.append(overall_row)

        for year, year_part in part.groupby("meta_year"):
            year_row = {
                "variant_name": variant_name,
                "score_name": "score_topk_raw_sum",
                "test_scope": str(year),
                "n_stages": year_part["stage_id"].nunique(),
            }
            for col in summary_metric_cols:
                year_row[col] = year_part[col].astype(float).mean()
            by_year_rows.append(year_row)

overall_metrics_df = pd.DataFrame(overall_rows)
by_year_metrics_df = pd.DataFrame(by_year_rows)

stage_metrics_df.to_csv(RESULT_DIR / "tabpfn_final_stage_metrics.csv", index=False)
overall_metrics_df.to_csv(RESULT_DIR / "tabpfn_final_metrics_overall.csv", index=False)
by_year_metrics_df.to_csv(RESULT_DIR / "tabpfn_final_metrics_by_year.csv", index=False)
class_distribution_df.to_csv(RESULT_DIR / "tabpfn_final_class_distribution.csv", index=False)
monotonicity_df.to_csv(RESULT_DIR / "tabpfn_final_probability_monotonicity_diagnostics.csv", index=False)
classification_metrics_df.to_csv(RESULT_DIR / "tabpfn_final_classification_metrics.csv", index=False)
confusion_matrices_df.to_csv(RESULT_DIR / "tabpfn_final_confusion_matrices.csv", index=False)
ordinal_confusion_matrix_df.to_csv(RESULT_DIR / "tabpfn_final_ordinal_confusion_matrix.csv", index=False)

print("=" * 88)
print("TABPFN 12-03: Finale Klassifikations- und Rankingmetriken")
print("=" * 88)
display(overall_metrics_df)
if not by_year_metrics_df.empty:
    print("Rankingmetriken nach Testjahr:")
    display(by_year_metrics_df)
if not classification_metrics_df.empty:
    print("Binäre Klassifikationsmetriken, Overall:")
    display(classification_metrics_df[classification_metrics_df["scope"] == "overall"].head(30))
if not confusion_matrices_df.empty:
    print("Konfusionsmatrizen, Overall:")
    display(confusion_matrices_df[confusion_matrices_df["scope"] == "overall"].head(40))
if not class_distribution_df.empty:
    print("Klassenverteilungen:")
    display(class_distribution_df.head(30))
if not monotonicity_df.empty:
    print("Monotonie-Diagnostics:")
    display(monotonicity_df.head(20))


TABPFN 12-03: Finale Klassifikations- und Rankingmetriken


,variant_name,score_name,test_scope,n_stages,ndcg_at_5,ndcg_at_10,ndcg_at_20,top_1_accuracy,top_5_accuracy,top_10_accuracy,top_20_accuracy,map
0,standard,score_topk_raw_sum,overall,112,0.354762,0.376148,0.430004,0.207207,0.414414,0.576577,0.666667,0.400759
1,thinking_medium_roc_auc,score_topk_raw_sum,overall,112,0.366026,0.367788,0.426508,0.243243,0.450450,0.522523,0.630631,0.396987


Rankingmetriken nach Testjahr:


,variant_name,score_name,test_scope,n_stages,ndcg_at_5,ndcg_at_10,ndcg_at_20,top_1_accuracy,top_5_accuracy,top_10_accuracy,top_20_accuracy,map
0,standard,score_topk_raw_sum,2024,57,0.372019,0.393663,0.438788,0.228070,0.421053,0.561404,0.649123,0.424177
1,standard,score_topk_raw_sum,2025,55,0.336879,0.357996,0.420900,0.185185,0.407407,0.592593,0.685185,0.376490
2,thinking_medium_roc_auc,score_topk_raw_sum,2024,57,0.378945,0.372763,0.428621,0.263158,0.438596,0.473684,0.614035,0.412056
3,thinking_medium_roc_auc,score_topk_raw_sum,2025,55,0.352636,0.362632,0.424318,0.222222,0.462963,0.574074,0.648148,0.381371


Binäre Klassifikationsmetriken, Overall:


...

,variant_name,scope,evaluation_name,target,threshold,threshold_source,actual_label,predicted_label,count
0,standard,overall,top5_native_0_5,top5,0.5,native_0_5,0,0,15189
1,standard,overall,top5_native_0_5,top5,0.5,native_0_5,0,1,2057
2,standard,overall,top5_native_0_5,top5,0.5,native_0_5,1,0,261
3,standard,overall,top5_native_0_5,top5,0.5,native_0_5,1,1,295
4,standard,overall,top10_native_0_5,top10,0.5,native_0_5,0,0,14357
5,standard,overall,top10_native_0_5,top10,0.5,native_0_5,0,1,2341
6,standard,overall,top10_native_0_5,top10,0.5,native_0_5,1,0,527
7,standard,overall,top10_native_0_5,top10,0.5,native_0_5,1,1,577
8,standard,overall,top20_native_0_5,top20,0.5,native_0_5,0,0,13032
9,standard,overall,top20_native_0_5,top20,0.5,native_0_5,0,1,2565


Klassenverteilungen:


,variant_name,scope,distribution_type,class_label,rows,row_share
0,standard,overall,predicted_rank_bucket,outside_top20,15562,0.874171
1,standard,overall,predicted_rank_bucket,rank_11_20,1120,0.062914
2,standard,overall,predicted_rank_bucket,rank_6_10,560,0.031457
3,standard,overall,predicted_rank_bucket,top5,560,0.031457
4,standard,overall,fh_seq_class_0_5,outside_top20,14113,0.792776
5,standard,overall,fh_seq_class_0_5,rank_11_20,1094,0.061454
6,standard,overall,fh_seq_class_0_5,rank_6_10,640,0.035951
7,standard,overall,fh_seq_class_0_5,top5,1955,0.109819
8,standard,year_2024,predicted_rank_bucket,outside_top20,7793,0.872383
9,standard,year_2024,predicted_rank_bucket,rank_11_20,570,0.063808


Monotonie-Diagnostics:


,variant_name,scope,rows,monotonic_rows,monotonicity_violation_rows,monotonicity_violation_rate,top5_gt_top10_rate,top10_gt_top20_rate
0,standard,overall,17802,8301,9501,0.533704,0.341366,0.323615
1,standard,year_2024,8933,4287,4646,0.520094,0.332923,0.312325
2,standard,year_2025,8869,4014,4855,0.547412,0.349870,0.334987
3,thinking_medium_roc_auc,overall,17802,9136,8666,0.486799,0.362712,0.241939
4,thinking_medium_roc_auc,year_2024,8933,4762,4171,0.466920,0.337513,0.235419
5,thinking_medium_roc_auc,year_2025,8869,4374,4495,0.506822,0.388093,0.248506


## Vergleichsoutputs und kompakte `data/models`-Dateien schreiben

Die finalen CSVs bleiben detailliert in `data/processed/tabpfn/03_final_three_targets/`. Für den Modellvergleich wird zusätzlich ein kompakter Standard-Output unter `data/models/` geschrieben.

Der Thinking-Vergleich wird getrennt behandelt: Die Statusdatei dokumentiert alle Varianten, aber die eigentliche Vergleichsdatei enthält nur Varianten mit vollständigen, plausibel echten Läufen für alle drei Targets.


In [9]:
# Thinking-Vergleich: nur vollständige und verifizierte Varianten werden in die Vergleichsdatei aufgenommen.
if not overall_metrics_df.empty:
    thinking_comparison_status = overall_metrics_df.merge(runtime_by_variant, on="variant_name", how="left")
    thinking_comparison_status["comparison_included"] = thinking_comparison_status["runtime_complete_for_comparison"].fillna(False).astype(bool)
    thinking_comparison_status["comparison_exclusion_reason"] = np.where(
        thinking_comparison_status["comparison_included"],
        "included",
        "missing_runtime_or_unverified_thinking",
    )
    thinking_comparison = thinking_comparison_status[thinking_comparison_status["comparison_included"]].copy()
else:
    thinking_comparison_status = pd.DataFrame()
    thinking_comparison = pd.DataFrame()

thinking_comparison_status.to_csv(RESULT_DIR / "tabpfn_final_thinking_comparison_status.csv", index=False)
thinking_comparison_status.to_csv(THINKING_DIR / "tabpfn_thinking_comparison_status.csv", index=False)
thinking_comparison.to_csv(RESULT_DIR / "tabpfn_final_thinking_comparison.csv", index=False)
thinking_comparison.to_csv(THINKING_DIR / "tabpfn_thinking_metrics.csv", index=False)

usage_rows = []
for _, runtime_row in runtime_by_variant.iterrows():
    usage_rows.append({
        "variant_name": runtime_row["variant_name"],
        "usage_status": "not_captured_in_notebook",
        "note": "Token/credit usage must be filled from client-specific usage APIs if available.",
    })
usage_df = pd.DataFrame(usage_rows)
usage_df.to_csv(RESULT_DIR / "tabpfn_final_usage.csv", index=False)
usage_df.to_csv(THINKING_DIR / "tabpfn_thinking_usage.csv", index=False)

# XGBoost-kompatibler Standard-Output: raw_prediction ist der kombinierte Top-K-Score.
standard_predictions = []
for combined in combined_predictions:
    if not combined.empty and combined["variant_name"].iloc[0] == "standard":
        standard_predictions.append(combined)

if standard_predictions:
    tabpfn_final_results = standard_predictions[0].copy()
    tabpfn_final_results["raw_prediction"] = tabpfn_final_results["score_topk_raw_sum"].astype(float)
    tabpfn_final_results["true_rank"] = tabpfn_final_results["actual_rank"]
    tabpfn_final_results["predicted_rank"] = tabpfn_final_results["rank_topk_raw_sum"]
    tabpfn_final_results["forced_ordinal_class"] = tabpfn_final_results["rank_forced_ordinal_class"]
    tabpfn_final_results = tabpfn_final_results.sort_values(["stage_id", "predicted_rank"]).reset_index(drop=True)
    model_result_cols = [
        "meta_year", "meta_name", "meta_current_team", "meta_race", "stage_nr", "stage_id",
        "raw_prediction", "true_rank", "predicted_rank", "forced_ordinal_class",
        "p_top5_raw", "p_top10_raw", "p_top20_raw", "score_topk_raw_sum",
        "predicted_rank_bucket", "fh_seq_class_0_5", "fh_seq_class_label_0_5",
        "is_cumulative_probability_monotonic", "violates_top5_le_top10", "violates_top10_le_top20",
        "ensemble_pred_top10_native_0_5", "ensemble_pred_top10_validated_threshold", "validated_top10_threshold",
    ]
    tabpfn_final_results[model_result_cols].to_pickle(MODEL_DIR / "tabpfn_final_results.pkl")
else:
    warnings.warn("Kein vollständiger Standardlauf vorhanden. data/models/tabpfn_final_results.pkl wird nicht geschrieben.")

# Kompakte Thesis-Tabelle mit den Pflichtspalten für den Modellvergleich.
comparison_cols = [
    "model", "variant_name", "score_name", "test_years", "n_stages",
    "ndcg_at_5", "ndcg_at_10", "ndcg_at_20",
    "winner_hit_at_1", "winner_hit_at_5", "winner_hit_at_10", "winner_hit_at_20",
    "map", "api_total_runtime_seconds", "runtime_complete_for_comparison",
]

if not overall_metrics_df.empty:
    tabpfn_metrics_comparison = overall_metrics_df.merge(runtime_by_variant, on="variant_name", how="left")
    tabpfn_metrics_comparison = tabpfn_metrics_comparison.rename(columns={
        "top_1_accuracy": "winner_hit_at_1",
        "top_5_accuracy": "winner_hit_at_5",
        "top_10_accuracy": "winner_hit_at_10",
        "top_20_accuracy": "winner_hit_at_20",
        "total_runtime_seconds": "api_total_runtime_seconds",
    })
    tabpfn_metrics_comparison["model"] = "TabPFN"
    tabpfn_metrics_comparison["test_years"] = "2024, 2025"
    tabpfn_metrics_comparison[comparison_cols].to_csv(MODEL_DIR / "tabpfn_final_metrics_comparison.csv", index=False)
else:
    pd.DataFrame(columns=comparison_cols).to_csv(MODEL_DIR / "tabpfn_final_metrics_comparison.csv", index=False)

output_artifacts = pd.DataFrame([
    {"artifact": "tabpfn_final_thinking_comparison.csv", "path": str(RESULT_DIR / "tabpfn_final_thinking_comparison.csv"), "exists": (RESULT_DIR / "tabpfn_final_thinking_comparison.csv").exists()},
    {"artifact": "tabpfn_final_thinking_comparison_status.csv", "path": str(RESULT_DIR / "tabpfn_final_thinking_comparison_status.csv"), "exists": (RESULT_DIR / "tabpfn_final_thinking_comparison_status.csv").exists()},
    {"artifact": "tabpfn_final_classification_metrics.csv", "path": str(RESULT_DIR / "tabpfn_final_classification_metrics.csv"), "exists": (RESULT_DIR / "tabpfn_final_classification_metrics.csv").exists()},
    {"artifact": "tabpfn_final_confusion_matrices.csv", "path": str(RESULT_DIR / "tabpfn_final_confusion_matrices.csv"), "exists": (RESULT_DIR / "tabpfn_final_confusion_matrices.csv").exists()},
    {"artifact": "tabpfn_final_results.pkl", "path": str(MODEL_DIR / "tabpfn_final_results.pkl"), "exists": (MODEL_DIR / "tabpfn_final_results.pkl").exists()},
    {"artifact": "tabpfn_standard_model_manifest.json", "path": str(STANDARD_MODEL_MANIFEST_PATH), "exists": STANDARD_MODEL_MANIFEST_PATH.exists()},
    {"artifact": "tabpfn_final_standard_models", "path": str(STANDARD_MODEL_DIR), "exists": STANDARD_MODEL_DIR.exists()},
    {"artifact": "tabpfn_final_metrics_comparison.csv", "path": str(MODEL_DIR / "tabpfn_final_metrics_comparison.csv"), "exists": (MODEL_DIR / "tabpfn_final_metrics_comparison.csv").exists()},
])

print("=" * 88)
print("TABPFN 12-03: Modellvergleichs- und Exportartefakte")
print("=" * 88)
display(output_artifacts)
if not thinking_comparison_status.empty:
    print("Thinking-Vergleichsstatus:")
    display(thinking_comparison_status)
if not thinking_comparison.empty:
    print("Verifizierter Thinking-Vergleich:")
    display(thinking_comparison)


TABPFN 12-03: Modellvergleichs- und Exportartefakte


,artifact,path,exists
0,tabpfn_final_thinking_comparison.csv,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
1,tabpfn_final_thinking_comparison_status.csv,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
2,tabpfn_final_classification_metrics.csv,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
3,tabpfn_final_confusion_matrices.csv,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
4,tabpfn_final_results.pkl,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
5,tabpfn_standard_model_manifest.json,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
6,tabpfn_final_standard_models,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
7,tabpfn_final_metrics_comparison.csv,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True


Thinking-Vergleichsstatus:


,variant_name,score_name,test_scope,n_stages,ndcg_at_5,ndcg_at_10,ndcg_at_20,top_1_accuracy,top_5_accuracy,top_10_accuracy,top_20_accuracy,map,targets_completed,targets_with_api_runtime,targets_expected,thinking_targets_verified,runtime_status,total_fit_seconds,total_predict_seconds,total_runtime_seconds,mean_runtime_seconds_per_target,runtime_complete_for_comparison,comparison_included,comparison_exclusion_reason
0,standard,score_topk_raw_sum,overall,112,0.354762,0.376148,0.430004,0.207207,0.414414,0.576577,0.666667,0.400759,3,3,3,3,api_run,17.043,94.931,112.389,37.463,True,True,included
1,thinking_medium_roc_auc,score_topk_raw_sum,overall,112,0.366026,0.367788,0.426508,0.243243,0.450450,0.522523,0.630631,0.396987,3,3,3,3,cache_hit,270.028,89.884,360.333,120.111,True,True,included


Verifizierter Thinking-Vergleich:


,variant_name,score_name,test_scope,n_stages,ndcg_at_5,ndcg_at_10,ndcg_at_20,top_1_accuracy,top_5_accuracy,top_10_accuracy,top_20_accuracy,map,targets_completed,targets_with_api_runtime,targets_expected,thinking_targets_verified,runtime_status,total_fit_seconds,total_predict_seconds,total_runtime_seconds,mean_runtime_seconds_per_target,runtime_complete_for_comparison,comparison_included,comparison_exclusion_reason
0,standard,score_topk_raw_sum,overall,112,0.354762,0.376148,0.430004,0.207207,0.414414,0.576577,0.666667,0.400759,3,3,3,3,api_run,17.043,94.931,112.389,37.463,True,True,included
1,thinking_medium_roc_auc,score_topk_raw_sum,overall,112,0.366026,0.367788,0.426508,0.243243,0.450450,0.522523,0.630631,0.396987,3,3,3,3,cache_hit,270.028,89.884,360.333,120.111,True,True,included


## Case Study: Tour de France 2025 Stage 12 und 16

Die Case Study vergleicht für beide Etappen die vorhergesagte Top20 direkt mit der echten Top20. Der Vergleich wird als Side-by-side-Tabelle geschrieben: Position 1 bis 20, predicted rider, actual rider, Score, echte Platzierung des Predictions-Fahrers und Trefferstatus.

Als Modell wird pro Variante explizit dokumentiert: `TabPFNClassifier`, die geladene Parameterauswahl aus `12-02`, die Variantenbezeichnung und der finale Score `score_topk_raw_sum = p_top5_raw + p_top10_raw + p_top20_raw`. Die Standardvariante ist der eigentliche Modellvergleich; Thinking erscheint nur, wenn die Variante vollständig vorhanden ist.


In [10]:
case_study_parts = []
case_study_comparison_rows = []
case_study_model_rows = []
expected_case_stage_ids = {"tour-de-france_2025_ST12", "tour-de-france_2025_ST16"}
case_study_score_name = "score_topk_raw_sum"
case_study_score_formula = "p_top5_raw + p_top10_raw + p_top20_raw"
case_study_model_family = selected.get("model_family", "TabPFNClassifier") if "selected" in globals() else "TabPFNClassifier"
case_study_model_params_json = json.dumps(selected.get("params", {}), ensure_ascii=True, sort_keys=True) if "selected" in globals() else "{}"

for combined in combined_predictions:
    # Die Pflicht-Case-Study fokussiert auf Tour de France 2025, Stage 12 und 16.
    case_mask = (
        (combined["meta_race"].astype(str) == "tour-de-france")
        & (combined["meta_year"].astype(int) == 2025)
        & (combined["stage_nr"].astype(int).isin([12, 16]))
    )
    cases = combined.loc[case_mask].copy()
    if cases.empty:
        continue

    if "score_name" not in cases.columns:
        cases["score_name"] = case_study_score_name

    for (variant_name, stage_id), stage in cases.groupby(["variant_name", "stage_id"]):
        model_used = f"{case_study_model_family} | variant={variant_name} | score={case_study_score_name}"
        model_role = "final_standard_model" if variant_name == "standard" else "optional_final_comparison_variant"
        stage_nr_value = int(stage["stage_nr"].iloc[0])
        race_value = str(stage["meta_race"].iloc[0])
        year_value = int(stage["meta_year"].iloc[0])

        case_study_model_rows.append({
            "stage_id": stage_id,
            "stage_nr": stage_nr_value,
            "variant_name": variant_name,
            "model_used": model_used,
            "model_family": case_study_model_family,
            "model_role": model_role,
            "model_params_json": case_study_model_params_json,
            "score_name": case_study_score_name,
            "score_formula": case_study_score_formula,
        })

        predicted = stage.sort_values(case_study_score_name, ascending=False).head(20).copy()
        predicted["view"] = "predicted_top20_by_score_topk_raw_sum"
        predicted["position"] = range(1, len(predicted) + 1)
        predicted["model_used"] = model_used
        predicted["model_family"] = case_study_model_family
        predicted["model_role"] = model_role
        predicted["score_formula"] = case_study_score_formula
        predicted["model_params_json"] = case_study_model_params_json

        actual = stage.sort_values("actual_rank", ascending=True).head(20).copy()
        actual["view"] = "actual_top20"
        actual["position"] = range(1, len(actual) + 1)
        actual["model_used"] = model_used
        actual["model_family"] = case_study_model_family
        actual["model_role"] = model_role
        actual["score_formula"] = case_study_score_formula
        actual["model_params_json"] = case_study_model_params_json

        case_study_parts.append(predicted)
        case_study_parts.append(actual)

        predicted_riders = set(predicted["meta_name"].astype(str))
        actual_riders = set(actual["meta_name"].astype(str))
        hit_count = len(predicted_riders & actual_riders)
        top20_precision = hit_count / len(predicted_riders) if predicted_riders else np.nan
        top20_recall = hit_count / len(actual_riders) if actual_riders else np.nan

        predicted_reset = predicted.reset_index(drop=True)
        actual_reset = actual.reset_index(drop=True)
        for position in range(1, 21):
            predicted_available = position <= len(predicted_reset)
            actual_available = position <= len(actual_reset)

            if predicted_available:
                predicted_row = predicted_reset.iloc[position - 1]
                predicted_rider = str(predicted_row["meta_name"])
                predicted_team = str(predicted_row["meta_current_team"])
                predicted_actual_rank = predicted_row["actual_rank"]
                predicted_score = float(predicted_row[case_study_score_name])
                predicted_stage_rank = int(predicted_row["rank_topk_raw_sum"])
                predicted_is_actual_top20 = predicted_rider in actual_riders
            else:
                predicted_rider = ""
                predicted_team = ""
                predicted_actual_rank = np.nan
                predicted_score = np.nan
                predicted_stage_rank = np.nan
                predicted_is_actual_top20 = False

            if actual_available:
                actual_row = actual_reset.iloc[position - 1]
                actual_rider = str(actual_row["meta_name"])
                actual_team = str(actual_row["meta_current_team"])
                actual_rank = actual_row["actual_rank"]
                actual_score = float(actual_row[case_study_score_name])
                actual_predicted_rank = int(actual_row["rank_topk_raw_sum"])
                actual_was_predicted_top20 = actual_rider in predicted_riders
            else:
                actual_rider = ""
                actual_team = ""
                actual_rank = np.nan
                actual_score = np.nan
                actual_predicted_rank = np.nan
                actual_was_predicted_top20 = False

            case_study_comparison_rows.append({
                "stage_id": stage_id,
                "stage_nr": stage_nr_value,
                "meta_year": year_value,
                "meta_race": race_value,
                "variant_name": variant_name,
                "model_used": model_used,
                "model_family": case_study_model_family,
                "model_role": model_role,
                "model_params_json": case_study_model_params_json,
                "score_name": case_study_score_name,
                "score_formula": case_study_score_formula,
                "comparison_position": position,
                "predicted_rider": predicted_rider,
                "predicted_team": predicted_team,
                "predicted_rank_by_model": predicted_stage_rank,
                "predicted_score_topk_raw_sum": predicted_score,
                "predicted_actual_rank": predicted_actual_rank,
                "predicted_is_actual_top20": predicted_is_actual_top20,
                "actual_rider": actual_rider,
                "actual_team": actual_team,
                "actual_rank": actual_rank,
                "actual_predicted_rank_by_model": actual_predicted_rank,
                "actual_score_topk_raw_sum": actual_score,
                "actual_was_predicted_top20": actual_was_predicted_top20,
                "top20_hit_count": hit_count,
                "top20_precision": top20_precision,
                "top20_recall": top20_recall,
            })

case_study_columns = [
    "stage_id", "stage_nr", "variant_name", "model_used", "model_family", "model_role",
    "model_params_json", "score_name", "score_formula", "view", "position", "rider", "team",
    "actual_rank", "predicted_rank", "score_topk_raw_sum",
    "predicted_rank_bucket", "fh_seq_class_label_0_5", "is_cumulative_probability_monotonic",
    "ensemble_pred_top10_native_0_5", "ensemble_pred_top10_validated_threshold", "validated_top10_threshold",
    "p_top5_raw", "p_top10_raw", "p_top20_raw",
    "actual_top5", "actual_top10", "actual_top20",
]

comparison_columns = [
    "stage_id", "stage_nr", "meta_year", "meta_race", "variant_name", "model_used", "model_family",
    "model_role", "model_params_json", "score_name", "score_formula", "comparison_position",
    "predicted_rider", "predicted_team", "predicted_rank_by_model", "predicted_score_topk_raw_sum",
    "predicted_actual_rank", "predicted_is_actual_top20",
    "actual_rider", "actual_team", "actual_rank", "actual_predicted_rank_by_model",
    "actual_score_topk_raw_sum", "actual_was_predicted_top20", "top20_hit_count", "top20_precision", "top20_recall",
]

model_columns = [
    "stage_id", "stage_nr", "variant_name", "model_used", "model_family", "model_role",
    "model_params_json", "score_name", "score_formula",
]

if case_study_parts:
    case_study_df = pd.concat(case_study_parts, ignore_index=True)
    case_study_df = case_study_df.rename(columns={
        "meta_name": "rider",
        "meta_current_team": "team",
        "rank_topk_raw_sum": "predicted_rank",
    })
    case_study_df = case_study_df[case_study_columns]
else:
    case_study_df = pd.DataFrame(columns=case_study_columns)

case_study_comparison_df = pd.DataFrame(case_study_comparison_rows, columns=comparison_columns)
case_study_model_df = pd.DataFrame(case_study_model_rows, columns=model_columns).drop_duplicates().reset_index(drop=True)

case_study_df.to_csv(CASE_STUDY_DIR / "tabpfn_case_study_tdf2025_stage12_16.csv", index=False)
case_study_comparison_df.to_csv(CASE_STUDY_DIR / "tabpfn_case_study_tdf2025_stage12_16_top20_comparison.csv", index=False)
case_study_model_df.to_csv(CASE_STUDY_DIR / "tabpfn_case_study_tdf2025_stage12_16_model_used.csv", index=False)

if not case_study_df.empty:
    found_case_stage_ids = set(case_study_df["stage_id"].astype(str).unique())
    missing_case_stage_ids = expected_case_stage_ids - found_case_stage_ids
    if missing_case_stage_ids:
        warnings.warn(f"Erwartete Case-Study-Stages fehlen: {sorted(missing_case_stage_ids)}")

summary_path = CASE_STUDY_DIR / "tabpfn_case_study_tdf2025_stage12_16_summary.md"
with open(summary_path, "w") as f:
    f.write("# TabPFN Case Study TDF 2025 Stage 12 und 16\n\n")
    if case_study_comparison_df.empty:
        f.write("Keine Case-Study-Predictions verfügbar. RUN_API=True setzen oder Prediction-Caches bereitstellen.\n")
    else:
        for (variant_name, stage_id), part in case_study_comparison_df.groupby(["variant_name", "stage_id"]):
            model_used = part["model_used"].iloc[0]
            model_params_json = part["model_params_json"].iloc[0]
            hit_count = int(part["top20_hit_count"].iloc[0])
            top20_precision = float(part["top20_precision"].iloc[0])
            top20_recall = float(part["top20_recall"].iloc[0])
            predicted_hits = part[part["predicted_is_actual_top20"] == True]["predicted_rider"].astype(str).tolist()
            missed_actual = part[part["actual_was_predicted_top20"] == False]["actual_rider"].astype(str).tolist()
            overestimated = part[part["predicted_is_actual_top20"] == False]["predicted_rider"].astype(str).tolist()
            winner_rows = part[pd.to_numeric(part["actual_rank"], errors="coerce") == 1]
            winner_text = "nicht verfügbar"
            if not winner_rows.empty:
                winner = winner_rows.iloc[0]
                winner_text = f"{winner['actual_rider']} mit vorhergesagtem Rang {winner['actual_predicted_rank_by_model']}"

            f.write(f"## {variant_name} - {stage_id}\n\n")
            f.write(f"Modell für diesen Vergleich: `{model_used}`.\n\n")
            f.write(f"Parameter aus `12-02`: `{model_params_json}`.\n\n")
            f.write(f"Score: `{case_study_score_name} = {case_study_score_formula}`.\n\n")
            f.write(f"- Top20-Treffer: {hit_count} von 20 predicted Top20-Fahrern; Precision {top20_precision:.3f}, Recall {top20_recall:.3f}.\n")
            f.write(f"- Echter Sieger: {winner_text}.\n")
            f.write(f"- Treffer in der vorhergesagten Top20: {', '.join(predicted_hits) if predicted_hits else 'keine'}.\n")
            f.write(f"- Überschätzte predicted Top20-Fahrer: {', '.join(overestimated) if overestimated else 'keine'}.\n")
            f.write(f"- Verpasste echte Top20-Fahrer: {', '.join(missed_actual) if missed_actual else 'keine'}.\n\n")
            f.write("Die vollständige Position-zu-Position-Tabelle steht in `tabpfn_case_study_tdf2025_stage12_16_top20_comparison.csv`.\n\n")

if case_study_comparison_df.empty:
    case_study_summary = pd.DataFrame([{"status": "empty", "message": "Keine vollständigen Case-Study-Predictions verfügbar."}])
else:
    case_study_summary = (
        case_study_comparison_df.groupby(["variant_name", "stage_id", "model_used"], dropna=False)
        .agg(
            rows=("comparison_position", "size"),
            top20_hit_count=("top20_hit_count", "first"),
            top20_precision=("top20_precision", "first"),
            top20_recall=("top20_recall", "first"),
        )
        .reset_index()
    )

print("=" * 88)
print("TABPFN 12-03: Case Study TDF 2025 Stage 12 und 16")
print("=" * 88)
display(case_study_model_df)
display(case_study_summary)
display(case_study_comparison_df.head(40))


TABPFN 12-03: Case Study TDF 2025 Stage 12 und 16


,stage_id,stage_nr,variant_name,model_used,model_family,model_role,model_params_json,score_name,score_formula
0,tour-de-france_2025_ST12,12,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw
1,tour-de-france_2025_ST16,16,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw
2,tour-de-france_2025_ST12,12,thinking_medium_roc_auc,TabPFNClassifier | variant=thinking_medium_roc...,TabPFNClassifier,optional_final_comparison_variant,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw
3,tour-de-france_2025_ST16,16,thinking_medium_roc_auc,TabPFNClassifier | variant=thinking_medium_roc...,TabPFNClassifier,optional_final_comparison_variant,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw


,variant_name,stage_id,model_used,rows,top20_hit_count,top20_precision,top20_recall
0,standard,tour-de-france_2025_ST12,TabPFNClassifier | variant=standard | score=sc...,20,8,0.4,0.4
1,standard,tour-de-france_2025_ST16,TabPFNClassifier | variant=standard | score=sc...,20,10,0.5,0.5
2,thinking_medium_roc_auc,tour-de-france_2025_ST12,TabPFNClassifier | variant=thinking_medium_roc...,20,6,0.3,0.3
3,thinking_medium_roc_auc,tour-de-france_2025_ST16,TabPFNClassifier | variant=thinking_medium_roc...,20,10,0.5,0.5


,stage_id,stage_nr,meta_year,meta_race,variant_name,model_used,model_family,model_role,model_params_json,score_name,score_formula,comparison_position,predicted_rider,predicted_team,predicted_rank_by_model,predicted_score_topk_raw_sum,predicted_actual_rank,predicted_is_actual_top20,actual_rider,actual_team,actual_rank,actual_predicted_rank_by_model,actual_score_topk_raw_sum,actual_was_predicted_top20,top20_hit_count,top20_precision,top20_recall
0,tour-de-france_2025_ST12,12,2025,tour-de-france,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw,1,Tadej Pogačar,UAE Team Emirates - XRG,1,2.877166,1.0,True,Tadej Pogačar,UAE Team Emirates - XRG,1.0,1,2.877166,True,8,0.4,0.4
1,tour-de-france_2025_ST12,12,2025,tour-de-france,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw,2,Jonas Vingegaard,Team Visma | Lease a Bike,2,2.852355,2.0,True,Jonas Vingegaard,Team Visma | Lease a Bike,2.0,2,2.852355,True,8,0.4,0.4
2,tour-de-france_2025_ST12,12,2025,tour-de-france,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw,3,Aleksandr Vlasov,Red Bull - BORA - hansgrohe,3,2.787662,34.0,False,Florian Lipowitz,Red Bull - BORA - hansgrohe,3.0,30,1.613841,False,8,0.4,0.4
3,tour-de-france_2025_ST12,12,2025,tour-de-france,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw,4,Matteo Jorgenson,Team Visma | Lease a Bike,4,2.771364,15.0,True,Tobias Halland Johannessen,Uno-X Mobility,4.0,17,2.126669,True,8,0.4,0.4
4,tour-de-france_2025_ST12,12,2025,tour-de-france,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw,5,Ben O'Connor,Team Jayco AlUla,5,2.746556,16.0,True,Oscar Onley,Team Picnic PostNL,5.0,18,2.073769,True,8,0.4,0.4
5,tour-de-france_2025_ST12,12,2025,tour-de-france,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw,6,Marc Hirschi,Tudor Pro Cycling Team,6,2.712906,57.0,False,Kévin Vauquelin,Arkéa - B&B Hotels,6.0,126,0.209306,False,8,0.4,0.4
6,tour-de-france_2025_ST12,12,2025,tour-de-france,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw,7,Remco Evenepoel,Soudal Quick-Step,7,2.692184,7.0,True,Remco Evenepoel,Soudal Quick-Step,7.0,7,2.692184,True,8,0.4,0.4
7,tour-de-france_2025_ST12,12,2025,tour-de-france,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw,8,Adam Yates,UAE Team Emirates - XRG,8,2.670213,23.0,False,Felix Gall,Decathlon AG2R La Mondiale Team,8.0,51,1.083441,False,8,0.4,0.4
8,tour-de-france_2025_ST12,12,2025,tour-de-france,standard,TabPFNClassifier | variant=standard | score=sc...,TabPFNClassifier,final_standard_model,"{""average_before_softmax"": true, ""balance_prob...",score_topk_raw_sum,p_top5_raw + p_top10_raw + p_top20_raw,9,Carlos Rodríguez,INEOS Grenadiers,9,2.643871,22.0,False,Primož Roglič,Red Bull - BORA - hansgrohe,9.0,16,2.232951,True,8,0.4,0.4
9,tour-de-france_2025_ST12,12,202

## Ergebnisinterpretation

Das Notebook erzeugt den finalen TabPFN-Standardoutput, optionale Thinking-Vergleiche, Klassifikationsmetriken, Konfusionsmatrizen, Rankingmetriken, Laufzeitvergleiche und die TDF-Case-Study. Alle finalen Rankings beruhen ausschließlich auf `score_topk_raw_sum`.

Die zentrale methodische Grenze bleibt sichtbar: Die drei TabPFN-Modelle sind getrennte binäre Classifier. Deshalb werden kumulative Wahrscheinlichkeiten nicht über Differenzen zu disjunkten Hauptwahrscheinlichkeiten umgedeutet. Monotonieverletzungen werden dokumentiert, aber nicht versteckt.

### Was aus `12-02` unverändert übernommen wurde

Die normale TabPFN-Konfiguration und der validierte Frank-&-Hall-Threshold stammen aus `tabpfn_selected_classifier_params.json`. `12-03` optimiert keinen Threshold auf 2024/2025. Wenn kein validierter Threshold vorhanden ist, bleibt die entsprechende Ensemble-Auswertung leer und das Notebook warnt sichtbar.

### Limitationen der finalen Interpretation

TabPFN liefert keine direkte Feature-Wichtigkeit wie ein EBM. Die Interpretation muss daher aus Scores, Klassenwahrscheinlichkeiten, Fehlerraten, Laufzeiten, Monotonie-Diagnostics und konkreten Case Studies entstehen. Genau deshalb dokumentiert dieses Notebook mehr Zwischenoutputs als ein reines Produktionsskript.
